In [1]:
import re
import pandas as pd
import numpy as np
import nltk
import pymorphy3
import optuna
import mlflow
import pickle
import implicit

from implicit.als import AlternatingLeastSquares
from scipy.sparse import coo_matrix, csr_matrix
from optuna.integration.mlflow import MLflowCallback
from mlflow.utils.mlflow_tags import MLFLOW_PARENT_RUN_ID
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import ndcg_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from nltk.corpus import stopwords
from mlxtend.plotting import plot_decision_regions
from tqdm import tqdm
from scipy.sparse import csr_matrix, vstack
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from gensim.utils import simple_preprocess
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import warnings
warnings.simplefilter('ignore', FutureWarning)

from sklearn import set_config
set_config(display='diagram')

In [2]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [3]:
TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000

EXPERIMENT_NAME = "hr-ai-scout"

mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")
mlflow.set_registry_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")

# Load data

In [4]:
df = pd.read_csv('/Users/user/Documents/Magistracy/year_project/hr-ai-scout/total_df.csv')
df.head()

,vacancy_id,vacancy_name,vacancy_area,vacancy_experience,vacancy_employment,vacancy_schedule,vacancy_salary_from,vacancy_salary_to,vacancy_salary_currency,vacancy_salary_gross,...,resume_education,resume_courses,resume_salary,resume_age,resume_total_experience,resume_experience_months,resume_location,resume_gender,resume_applicant_status,target
0,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,300000.0,NaN,RUR,False,...,['Казанский Авиационный Институт'],NaN,NaN,65.0,19 лет,228.0,Москва,Мужчина,Рассматривает предложения,1
1,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,300000.0,NaN,RUR,False,...,"['ООО ""Открытый Учебный Центр СофтБаланс"", г. ...","['ООО ""Открытый Учебный Центр СофтБаланс"", г. ...",NaN,43.0,17 лет 4 месяца,208.0,Москва,Мужчина,Рассматривает предложения,1
2,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,300000.0,NaN,RUR,False,...,['Орский государственный педагогический инстит...,NaN,200 000 ₽ на руки,52.0,30 лет,360.0,Москва,Женщина,NaN,1
3,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,300000.0,NaN,RUR,False,...,['Красноярский государственный университет'],NaN,500 000 ₽ на руки,56.0,29 лет 8 месяцев,356.0,Красноярск,Мужчина,Рассматривает предложения,1
4,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,300000.0,NaN,RUR,False,...,['Белоруский Гос. Университет Информатики и Ра...,"['SAP CIS, SAP XI', 'Школа Логистики МАДИ', 'S...",NaN,48.0,25 лет 1 месяц,301.0,Moscow,Male,NaN,1


# Preprocessing

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">
В первую очередь уберем строки, где пропущены все ключевые значения в резюме:
</div>

In [5]:
t1 = df.shape[0]
df = df.dropna(subset= ["resume_education",
                        "resume_last_experience_description",
                        "resume_last_position",
                        "resume_last_company_experience_period",
                        "resume_total_experience",
                        "resume_experience_months",
                        "resume_location",
                        "resume_specialization",
                        # "resume_gender",
                        # "resume_title"
                       ], how="all")
t2 = df.shape[0]
print('Удалено ', t1 - t2 ,' строки')

Удалено  84  строки


<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">
Удалим еще те строки, где случился технический сбой в парсинге, где у кандидата общий опыт есть, а последний опыт не указан (и наоборот):
</div>

In [6]:
t1 = df.shape[0]
df = df.loc[~(df["resume_total_experience"].notna()
        & df["resume_last_experience_description"].isna()
        & df["resume_last_position"].isna())]
t2 = df.shape[0]
print('Удалено ', t1 - t2 ,' строк')

Удалено  1543  строк


In [7]:
t1 = df.shape[0]
df = df.loc[~(df["resume_total_experience"].isna()
        & df["resume_last_experience_description"].notna()
        & df["resume_last_position"].notna())]
t2 = df.shape[0]
print('Удалено ', t1 - t2 ,' строк')

Удалено  0  строк


<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">
Посмотрим на пропуски отдельно по категориальным и числовым признакам.
</div>

In [8]:
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(include=['object']).columns

In [9]:
df[cat_cols] = df[cat_cols].fillna('NDT')

In [10]:
df.loc[df['resume_experience_months'].isna(), 'resume_last_experience_description'].unique()

array(['NDT'], dtype=object)

In [11]:
df['resume_age'] = df['resume_age'].fillna(df['resume_age'].mean())
df['resume_experience_months'] = df['resume_experience_months'].fillna(0)

In [12]:
df = df.drop(['vacancy_salary_to', 'vacancy_salary_from',
              'vacancy_salary_currency', 'vacancy_salary_gross'], axis=1)

In [13]:
df.loc[df['resume_last_company_experience_period'] == 'NDT', 'resume_last_experience_description'].unique()

array(['NDT'], dtype=object)

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">
Преобразуем сначала ожидаемые зарплаты
</div>

In [14]:
df['resume_salary_split'] = df['resume_salary'].apply(lambda x: x.split())

df['salary_int'] = df['resume_salary_split'].apply(
    lambda x: int(''.join(part for part in x if re.fullmatch(r'\d+', part)))
              if any(re.fullmatch(r'\d+', part) for part in x)
              else np.nan
)

currency_symbols = ['₽', '$', '€', '₴', '₸', '₼', '₾', 'Br', "so'm"]

rates_rub = {
    "₽": 1.0,
    "$": 80.85,
    "€": 94.14,
    "₴": 1.94,
    "₸": 0.150,
    "₼": 47.8,
    "₾": 33.5,
    "Br": 28.7,
    "so'm": 0.0068
}

df['currency_symbol'] = df['resume_salary_split'].apply(
    lambda x: next((sym for sym in x if sym in currency_symbols), np.nan)
)

df['salary_converted'] = (df['salary_int'] * df['currency_symbol'].map(rates_rub)).fillna(0)

df['resume_salary'] = df['salary_converted']

df = df.drop(['resume_salary_split', 'salary_int', 'currency_symbol', 'salary_converted'], axis=1)

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">
Добавим дополнительный столбец с опытом работы в последней компании в месяцах для удобства
</div>

In [15]:
def experience_to_months(experience_text):
    months = 0
    # Опыт в годах
    years_match = re.search(r'(\d+)\s*год', experience_text)
    if years_match:
        months += int(years_match.group(1)) * 12

    years_match = re.search(r'(\d+)\s*лет', experience_text)
    if years_match:
        months += int(years_match.group(1)) * 12

    # Опыт в месяцах
    months_match = re.search(r'(\d+)\s*месяц', experience_text)
    if months_match:
        months += int(months_match.group(1))

    return months if months > 0 else np.nan

In [16]:
df['resume_last_company_experience_months'] = df['resume_last_company_experience_period'].apply(experience_to_months)

In [17]:
df.loc[df['resume_last_company_experience_period'] == 'NDT', 'resume_last_experience_description'].unique()

array(['NDT'], dtype=object)

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">
Т.к. в названии компании стоит NDT, можно столбец resume_last_company_experience_months заполнять нулевыми значениями.
</div>

In [18]:
df['resume_last_company_experience_months'] = df['resume_last_company_experience_months'].fillna(0)

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

- Ограничим выбросы по зарплате, потому что ровно одно значение по ожидаемой заработоной плате = 999,999,999 (смешно, но нет)

- Ограничим опыт общий и внутри одной компании до 720 месяцев (60 лет, ничего себе уже)

- Уберем возраст > 90, не ждем, что эти кандидаты находятся в поиске вакансии
</div>

In [19]:
df = df[~(df.resume_salary > 1e7)]
df.loc[df['resume_experience_months'] > 720, 'resume_experience_months'] = 720
df.loc[df['resume_last_company_experience_months'] > 720, 'resume_last_company_experience_months'] = 720
df = df[~(df.resume_age > 90)]

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

- Также уберем строки, где последний опыт кандидата больше, чем общий

- И где общий опыт кандидата +16 лет больше чем возраст (хоть так)

</div>

In [20]:
df = df[~(df.resume_experience_months < df.resume_last_company_experience_months)]
df = df[~(df.resume_age < (df.resume_experience_months // 12) + 16)]

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

Заменим текущий формат разброса полов в датасете на унифицированный

</div>

In [21]:
gender_map = {
    'Мужчина': 'Мужчина',
    'Male': 'Мужчина',
    'Женщина': 'Женщина',
    'Female': 'Женщина'
}

df['resume_gender'] = df['resume_gender'].apply(lambda x: gender_map[x] if x in gender_map else 'Неизвестно')

# Feature engineering

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">
Добавим признак матчинга локации вакансии и резюме

</div>

In [22]:
df['location_matching'] = df.apply(lambda row: 1 if row['vacancy_area'] == row['resume_location'] else 0, axis=1)

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

Сделаем новый признак, а именно посчитаем количество навыков кандидата, которые указаны в вакансии.

</div>

In [23]:
def resume_skill_count_in_vacancy(row):
    count = 0
    skill_list = row['resume_skills'].replace('[', '').replace(']', '').replace("'", "").split(', ')
    for i in skill_list:
        if i in row['vacancy_description']:
            count += 1
    return count

df['resume_skill_count_in_vacancy'] = df.apply(resume_skill_count_in_vacancy, axis=1)

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

Также посчитаем долю слов из последней должности в резюме, которые указаны в вакансии.

</div>

In [24]:
def last_position_in_vacancy(row):
    bow = []
    seps = [' ', '-', '_']
    for sep in seps:
        bow += row['resume_last_position'].split(sep=sep)
        bow = list(set(bow))
    
    c = 0
    for word in bow:
        if word in row['vacancy_description']:
            c +=1
    
    return c / len(bow)

In [25]:
df['last_position_in_vacancy'] = df.apply(last_position_in_vacancy, axis=1)

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

Теперь закодируем описание вакансии и последнего опыта работы и сравним через косинусное расстояние.

</div>

In [26]:
def preprocess_data(df):
    """Обработка пропущенных значений в текстовых полях"""
    print("Проверка пропущенных значений...")
    print(f"Пропуски в vacancy_description: {df['vacancy_description'].isna().sum()}")
    print(f"Пропуски в resume_last_experience_description: {df['resume_last_experience_description'].isna().sum()}")
    
    # Заполняем пропуски пустыми строками
    df['vacancy_description'] = df['vacancy_description'].fillna('')
    df['resume_last_experience_description'] = df['resume_last_experience_description'].fillna('')
    
    # Проверяем, что все значения теперь строковые
    df['vacancy_description'] = df['vacancy_description'].astype(str)
    df['resume_last_experience_description'] = df['resume_last_experience_description'].astype(str)
    
    return df

In [27]:
def save_results(df, output_file):
    """Сохранение результатов в CSV файл"""
    df.to_csv(output_file, index=False, encoding='utf-8')
    print(f"Результаты сохранены в файл: {output_file}")

In [28]:
def calculate_cosine_similarity(embeddings1, embeddings2):
    """Вычисление косинусного сходства между двумя наборами эмбеддингов"""
    similarities = []
    
    for i in tqdm(range(embeddings1.shape[0])):
        emb1_row = embeddings1[i]
        emb2_row = embeddings2[i]
        
        similarity = cosine_similarity(emb1_row, emb2_row)[0][0]
        similarities.append(similarity)
    
    return similarities

In [29]:
warnings.filterwarnings('ignore')

try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

try:
    nltk.data.find('taggers/averaged_perceptron_tagger_ru')
except LookupError:
    nltk.download('averaged_perceptron_tagger_ru')

try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet')

morph = pymorphy3.MorphAnalyzer()

[nltk_data] Downloading package wordnet to /Users/user/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [30]:
def lemmatize_russian(tokens):
    """Лемматизация русских слов"""
    lemmas = []
    for token in tokens:
        parsed = morph.parse(token)[0]  # Берем самый вероятный разбор
        lemmas.append(parsed.normal_form)
    return lemmas

In [31]:
def tokenize_and_lemmatize(text):
    """Токенизация текста с лемматизацией и удалением стоп-слов"""
    tokens = simple_preprocess(text, deacc=True, min_len=2)
    stop_words = set(stopwords.words('russian') + stopwords.words('english'))
    tokens = [token for token in tokens if token not in stop_words]
    lemmatized_tokens = lemmatize_russian(tokens)
    
    return lemmatized_tokens

In [32]:
def get_tfidf_embeddings(texts, vectorizer=None, fit=True):
    """Создание TF-IDF эмбеддингов для списка текстов с лемматизацией"""
    if fit:
        vectorizer = TfidfVectorizer(
            max_features=5000,
            min_df=2,
            max_df=0.8,
            ngram_range=(1, 2),
            tokenizer=tokenize_and_lemmatize,
            token_pattern=None,
            lowercase=False  # Уже сделано в токенизации
        )
        embeddings = vectorizer.fit_transform(texts)
    else:
        embeddings = vectorizer.transform(texts)
    
    return embeddings, vectorizer

In [33]:
def get_tfidf_vacancy_embeddings(df, vectorizer=None):
    """Создание эмбеддингов для уникальных вакансий с лемматизацией"""
    unique_vacancies = df[['vacancy_id', 'vacancy_description']].drop_duplicates()
    
    unique_embeddings, vectorizer = get_tfidf_embeddings(
        unique_vacancies['vacancy_description'].tolist(), 
        vectorizer=vectorizer, 
        fit=(vectorizer is None)
    )
    
    vacancy_embedding_dict = dict(zip(unique_vacancies['vacancy_id'], unique_embeddings))
    
    rows = []
    for vid in df['vacancy_id']:
        rows.append(vacancy_embedding_dict[vid])
    
    all_vacancy_embeddings = vstack(rows)
    
    return all_vacancy_embeddings, vectorizer

In [34]:
def process_similarity_scores_tfidf(df, vectorizer=None, fit=True):
    """Функция для вычисления схожести с использованием TF-IDF и лемматизации"""
    df = preprocess_data(df)
    
    print("Создание TF-IDF эмбеддингов для описаний опыта в резюме...")
    experience_embeddings, tfidf_vectorizer = get_tfidf_embeddings(df['resume_last_experience_description'].tolist(), vectorizer=vectorizer, fit=fit)
    
    print("Создание TF-IDF эмбеддингов для описаний вакансий...")
    vacancy_embeddings, _ = get_tfidf_vacancy_embeddings(df, vectorizer=tfidf_vectorizer)
    
    print("Вычисление косинусного сходства...")
    similarity_scores = calculate_cosine_similarity(vacancy_embeddings, experience_embeddings)
    
    df['similarity_score_tfidf'] = similarity_scores
    
    return df, tfidf_vectorizer

In [35]:
try:
    df_tfidf = pd.read_csv('description_df_with_scores_tfidf.csv')
except:
    df_tfidf = process_similarity_scores_tfidf(df.copy())
    save_results(df_tfidf, 'description_df_with_scores_tfidf.csv')

In [36]:
df = df.merge(df_tfidf)

In [37]:
df.head()

,vacancy_id,vacancy_name,vacancy_area,vacancy_experience,vacancy_employment,vacancy_schedule,vacancy_description,resume_id,resume_title,resume_specialization,...,resume_location,resume_gender,resume_applicant_status,target,resume_last_company_experience_months,location_matching,resume_skill_count_in_vacancy,last_position_in_vacancy,resume_skill_count,similarity_score_tfidf
0,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,"Привет!.redev — технологическая компания, созд...",6969174,ABAP-разработчик,"['Программист, разработчик']",...,Москва,Мужчина,Рассматривает предложения,1,76.0,1,3,0.666667,3,0.284047
1,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,"Привет!.redev — технологическая компания, созд...",9100077,"ABAP разработчик - SAP HCM, CRM, S/4HANA ERP(F...","['Программист, разработчик']",...,Москва,Мужчина,Рассматривает предложения,1,8.0,1,2,0.500000,2,0.308726
2,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,"Привет!.redev — технологическая компания, созд...",32644957,Разработчик ABAP,"['Программист, разработчик']",...,Москва,Женщина,NDT,1,136.0,1,1,0.000000,1,0.510093
3,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,"Привет!.redev — технологическая компания, созд...",27220466,ABAP-разработчик,"['Программист, разработчик']",...,Красноярск,Мужчина,Рассматривает предложения,1,135.0,0,2,0.333333,2,0.301062
4,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,"Привет!.redev — технологическая компания, созд...",7532708,ABAP разработчик. Senior ABAP Developer. SAP T...,"['Programmer, developer']",...,Moscow,Мужчина,NDT,1,0.0,0,2,0.600000,2,0.075429


# Train-test split

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

Добавим новые признаки в обучение и тестирование

</div>

In [38]:
features = [
    'vacancy_area',
    'vacancy_experience',
    'vacancy_employment', 
    'vacancy_schedule',
    # 'resume_specialization',
    # 'resume_education', 
    # 'resume_courses', 
    'resume_salary',
    'resume_age', 
    'resume_experience_months',
    'resume_location',
    'resume_gender', 
    'resume_applicant_status', 
    'resume_last_company_experience_months', 
    'location_matching',
    'resume_skill_count_in_vacancy',
    'last_position_in_vacancy',
    'similarity_score_tfidf'
]
df[features]

,vacancy_area,vacancy_experience,vacancy_employment,vacancy_schedule,resume_salary,resume_age,resume_experience_months,resume_location,resume_gender,resume_applicant_status,resume_last_company_experience_months,location_matching,resume_skill_count_in_vacancy,last_position_in_vacancy,similarity_score_tfidf
0,Москва,Более 6 лет,Полная занятость,Удаленная работа,0.0,65.000000,228.0,Москва,Мужчина,Рассматривает предложения,76.0,1,3,0.666667,0.284047
1,Москва,Более 6 лет,Полная занятость,Удаленная работа,0.0,43.000000,208.0,Москва,Мужчина,Рассматривает предложения,8.0,1,2,0.500000,0.308726
2,Москва,Более 6 лет,Полная занятость,Удаленная работа,200000.0,52.000000,360.0,Москва,Женщина,NDT,136.0,1,1,0.000000,0.510093
3,Москва,Более 6 лет,Полная занятость,Удаленная работа,500000.0,56.000000,356.0,Красноярск,Мужчина,Рассматривает предложения,135.0,0,2,0.333333,0.301062
4,Москва,Более 6 лет,Полная занятость,Удаленная работа,0.0,48.000000,301.0,Moscow,Мужчина,NDT,0.0,0,2,0.600000,0.075429
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
321637,Москва,От 3 до 6 лет,Полная занятость,Удаленная работа,242550.0,66.000000,521.0,Санкт-Петербург,Женщина,NDT,270.0,0,0,0.166667,0.072670
321638,Москва,От 3 до 6 лет,Полная занятость,Удаленная работа,0.0,40.000000,213.0,Москва,Мужчина,Активно ищет работу,35.0,1,0,0.000000,0.000000
321639,Москва,От 3 до 6 лет,Полная занятость,Удаленная работа,80000.0,44.060813,121.0,Москва,Мужчина,NDT,44.0,1,0,0.200000,0.047398
321640,Москва,От 3 до 6 лет,Полная занятость,Удаленная работа,0.0,32.000000,117.0,Москва,Женщина,NDT,96.0,1,0,0.200000,0.029086


In [39]:
numeric_features = df[features].select_dtypes(include=np.number).columns
categorical_features = df[features].select_dtypes(exclude=np.number).columns

In [40]:
X = df[features].copy()
y = df['target'].copy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

In [41]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((257313, 15), (64329, 15), (257313,), (64329,))

In [42]:
def calculate_metrics(df_test: pd.DataFrame) -> pd.DataFrame:
    ndcg_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []
    vacancy_ids = df_test['vacancy_id'].unique()
    
    for vacancy_id in vacancy_ids:
        mask = df_test['vacancy_id'] == vacancy_id
        y_true = df_test.loc[mask, 'target'].values
        y_score = df_test.loc[mask, 'y_pred_proba'].values
        
        if len(y_true) <= 1:
            continue
        
        y_true_2d = y_true.reshape(1, -1)
        y_score_2d = y_score.reshape(1, -1)
        
        ndcg = ndcg_score(y_true_2d, y_score_2d)
        ndcg_scores.append(ndcg)
        
        y_pred_binary = (y_score >= 0.5).astype(int)
        
        precision = precision_score(y_true, y_pred_binary, zero_division=0)
        recall = recall_score(y_true, y_pred_binary, zero_division=0)
        f1 = f1_score(y_true, y_pred_binary, zero_division=0)
        
        precision_scores.append(precision)
        recall_scores.append(recall)
        f1_scores.append(f1)
    
    if ndcg_scores:
        print(f"Средний NDCG: {np.mean(ndcg_scores):.4f}")
        print(f"Средний Precision: {np.mean(precision_scores):.4f}")
        print(f"Средний Recall: {np.mean(recall_scores):.4f}")
        print(f"Средний F1-Score: {np.mean(f1_scores):.4f}")

        return np.mean(ndcg_scores), np.mean(precision_scores), np.mean(recall_scores), np.mean(f1_scores)
    else:
        print("Недостаточно данных для расчета метрик")

        return None, None, None, None

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

Обучим с подбором гиперпараметров `LogisticRegression`, как бейзлайн для сравнения с нелинейными моделями, `RandomForestClassifier`, как разновидность бэггинга, и три разные вида градиентного бустинга: `XGBClassifier`, `LGBMClassifier` и `CatBoostClassifier`.

</div>

# LogisticRegression

In [43]:
def objective(trial: optuna.Trial) -> float:
    params = {
        'model__C': trial.suggest_float('C', 1e-4, 1e4, log=True),
        'model__penalty': trial.suggest_categorical('penalty', ['l1', 'l2']),
        'model__solver': trial.suggest_categorical('solver', ['liblinear', 'saga'])
    }
    
    pipeline_lr_optuna = Pipeline([
        ('preprocessing', ColumnTransformer([
            ('numeric_scaling', StandardScaler(), numeric_features),
            ('categorical_encoding', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
        ])),
        ('model', LogisticRegression(random_state=RANDOM_STATE, class_weight='balanced'))
    ])
    
    pipeline_lr_optuna.set_params(**params)
    
    kfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    
    for train_idx, val_idx in kfold.split(X_train, y_train):
        X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        pipeline_lr_optuna.fit(X_fold_train, y_fold_train)
        y_pred_proba = pipeline_lr_optuna.predict_proba(X_fold_val)
        

        df_val = df.loc[X_fold_val.index].copy()
        df_val['y_pred_proba'] = y_pred_proba[:, 1]
        
        ndcg, _, _, _ = calculate_metrics(df_val)

        trial.set_user_attr('pipeline_params', params)
    
    return ndcg

In [44]:
try:
    experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
except:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)

RUN_NAME_OPTUNE = 'LogisticRegression_optuna'

with mlflow.start_run(run_name=RUN_NAME_OPTUNE, experiment_id=experiment_id) as run:
    run_id = run.info.run_id

🏃 View run LogisticRegression_optuna at: http://127.0.0.1:5000/#/experiments/1/runs/a485f72d148b40f19596ab478111238e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [45]:
STUDY_DB_NAME = "sqlite:///local.study.db"
STUDY_NAME = "LogisticRegression_optuna"

mlflc = MLflowCallback(
    tracking_uri=f'http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}',
    metric_name="NDCG",
    create_experiment=False,
    mlflow_kwargs={'experiment_id': experiment_id, 'tags': {MLFLOW_PARENT_RUN_ID: run_id}}
)

In [46]:
study = optuna.create_study(direction='maximize', 
                            sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
                            study_name=STUDY_NAME,
                            storage=STUDY_DB_NAME,
                            load_if_exists=True)
study.optimize(objective, 
               n_trials=10, 
               callbacks=[mlflc]
              )
best_params_optuna = study.best_params

print(f"Number of finished trials: {len(study.trials)}")
print(f"Best params: {best_params_optuna}")

[I 2026-05-02 17:45:43,763] Using an existing study with name 'LogisticRegression_optuna' instead of creating a new one.


Средний NDCG: 0.8294
Средний Precision: 0.5720
Средний Recall: 0.7882
Средний F1-Score: 0.6322
Средний NDCG: 0.8174
Средний Precision: 0.5638
Средний Recall: 0.7777
Средний F1-Score: 0.6220


[I 2026-05-02 17:46:04,189] Trial 41 finished with value: 0.8257821305336338 and parameters: {'C': 0.013254707147861793, 'penalty': 'l1', 'solver': 'liblinear'}. Best is trial 30 with value: 0.8272130448068064.


Средний NDCG: 0.8258
Средний Precision: 0.5709
Средний Recall: 0.7823
Средний F1-Score: 0.6300
🏃 View run 41 at: http://127.0.0.1:5000/#/experiments/1/runs/11ef16ef253845958847e942bc7f2f40
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8304
Средний Precision: 0.5689
Средний Recall: 0.7890
Средний F1-Score: 0.6305
Средний NDCG: 0.8193
Средний Precision: 0.5629
Средний Recall: 0.7798
Средний F1-Score: 0.6224


[I 2026-05-02 17:46:41,809] Trial 42 finished with value: 0.826797257189189 and parameters: {'C': 0.21739643219798532, 'penalty': 'l1', 'solver': 'saga'}. Best is trial 30 with value: 0.8272130448068064.


...

Средний NDCG: 0.8262
Средний Precision: 0.5696
Средний Recall: 0.7818
Средний F1-Score: 0.6293
🏃 View run 48 at: http://127.0.0.1:5000/#/experiments/1/runs/b4a3d530469548d1850c2299747807b7
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8302
Средний Precision: 0.5689
Средний Recall: 0.7892
Средний F1-Score: 0.6305
Средний NDCG: 0.8192
Средний Precision: 0.5621
Средний Recall: 0.7796
Средний F1-Score: 0.6220


[I 2026-05-02 17:51:38,004] Trial 49 finished with value: 0.8271220916127241 and parameters: {'C': 0.5767709308120591, 'penalty': 'l1', 'solver': 'saga'}. Best is trial 30 with value: 0.8272130448068064.


Средний NDCG: 0.8271
Средний Precision: 0.5717
Средний Recall: 0.7820
Средний F1-Score: 0.6306
🏃 View run 49 at: http://127.0.0.1:5000/#/experiments/1/runs/9a06ed06b7af4c8e885ca716a2f55826
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8302
Средний Precision: 0.5689
Средний Recall: 0.7892
Средний F1-Score: 0.6305
Средний NDCG: 0.8191
Средний Precision: 0.5624
Средний Recall: 0.7797
Средний F1-Score: 0.6221


[I 2026-05-02 17:52:24,814] Trial 50 finished with value: 0.8270525542561754 and parameters: {'C': 0.45123756883616173, 'penalty': 'l1', 'solver': 'saga'}. Best is trial 30 with value: 0.8272130448068064.


Средний NDCG: 0.8271
Средний Precision: 0.5717
Средний Recall: 0.7822
Средний F1-Score: 0.6308
🏃 View run 50 at: http://127.0.0.1:5000/#/experiments/1/runs/d142ea5033224334af787b833f17fb97
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Number of finished trials: 51
Best params: {'C': 0.18851801168357057, 'penalty': 'l1', 'solver': 'liblinear'}


In [47]:
study.best_trial.user_attrs['pipeline_params']

{'model__C': 0.18851801168357057,
 'model__penalty': 'l1',
 'model__solver': 'liblinear'}

In [48]:
pipeline_lr_best_optuna = Pipeline([
    ('preprocessing', ColumnTransformer([
        ('numeric_scaling', StandardScaler(), numeric_features),
        ('categorical_encoding', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
    ])),
    ('model', LogisticRegression(random_state=RANDOM_STATE))
])

pipeline_lr_best_optuna.set_params(**study.best_trial.user_attrs['pipeline_params'])
pipeline_lr_best_optuna.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric_scaling', ...), ('categorical_encoding', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of

In [49]:
y_pred_proba = pipeline_lr_best_optuna.predict_proba(X_test)

df_test = df.loc[X_test.index].copy()
df_test['y_pred_proba'] = y_pred_proba[:, 1]

In [50]:
ndcg_lr, precision_lr, recall_lr, f1_lr = calculate_metrics(df_test)

metrics_lr = {
    'NDCG': ndcg_lr,
    'Precision': precision_lr,
    'Recall': recall_lr,
    'F1': f1_lr
}

Средний NDCG: 0.7515
Средний Precision: 0.6386
Средний Recall: 0.5985
Средний F1-Score: 0.6021


In [51]:
RUN_NAME = "best_optuna_lr" 
REGISTRY_MODEL_NAME = "best_optuna_lr"

In [52]:
signature = mlflow.models.infer_signature(X_test, y_test)
input_example = X_test[:10]
code_paths = ["ML_Experiments.ipynb"]

try:
    experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
except:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id

    lr_info = mlflow.sklearn.log_model(sk_model=pipeline_lr_best_optuna, 
                                       artifact_path='best_optuna_lr',
                                       registered_model_name=REGISTRY_MODEL_NAME,
                                       input_example=input_example,
                                       code_paths=code_paths,
                                       await_registration_for=60
                                      )
    mlflow.log_metrics(metrics_lr)
    mlflow.log_params(best_params_optuna)

Registered model 'best_optuna_lr' already exists. Creating a new version of this model...
2026/05/02 17:52:39 INFO mlflow.store.model_registry.abstract_store: Waiting up to 60 seconds for model version to finish creation. Model name: best_optuna_lr, version 5


🏃 View run best_optuna_lr at: http://127.0.0.1:5000/#/experiments/1/runs/e89dfb9ba5654f7ea99734663ce7c3ca
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


Created version '5' of model 'best_optuna_lr'.


# RandomForestClassifier

In [53]:
def objective(trial: optuna.Trial) -> float:
    params = {
        'model__n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=50),
        'model__max_depth': trial.suggest_int('max_depth', 3, 30),
        'model__min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'model__min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20)
    }
    
    pipeline_rf = Pipeline([
        ('preprocessing', ColumnTransformer([
            ('numeric_scaling', StandardScaler(), numeric_features),
            ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
        ])),
        ('model', RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced'))
    ])
    
    pipeline_rf.set_params(**params)
    
    kfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    
    for train_idx, val_idx in kfold.split(X_train, y_train):
        X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        pipeline_rf.fit(X_fold_train, y_fold_train)
        y_pred_proba = pipeline_rf.predict_proba(X_fold_val)
        

        df_val = df.loc[X_fold_val.index].copy()
        df_val['y_pred_proba'] = y_pred_proba[:, 1]
        
        ndcg, _, _, _ = calculate_metrics(df_val)

        trial.set_user_attr('pipeline_params', params)
    
    return ndcg

In [54]:
try:
    experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
except:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)

RUN_NAME_OPTUNE = 'RandomForestClassifier_optuna'

with mlflow.start_run(run_name=RUN_NAME_OPTUNE, experiment_id=experiment_id) as run:
    run_id = run.info.run_id

🏃 View run RandomForestClassifier_optuna at: http://127.0.0.1:5000/#/experiments/1/runs/aaaf5c21225f44a0a7b1d538767d1068
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [55]:
STUDY_DB_NAME = "sqlite:///local.study.db"
STUDY_NAME = "RandomForestClassifier_optuna"

mlflc = MLflowCallback(
    tracking_uri=f'http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}',
    metric_name="NDCG",
    create_experiment=False,
    mlflow_kwargs={'experiment_id': experiment_id, 'tags': {MLFLOW_PARENT_RUN_ID: run_id}}
)

In [56]:
study = optuna.create_study(direction='maximize', 
                            sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
                            study_name=STUDY_NAME,
                            storage=STUDY_DB_NAME,
                            load_if_exists=True)
study.optimize(objective, 
               n_trials=10, 
               callbacks=[mlflc]
              )
best_params_optuna = study.best_params

print(f"Number of finished trials: {len(study.trials)}")
print(f"Best params: {best_params_optuna}")

[I 2026-05-02 17:52:39,540] Using an existing study with name 'RandomForestClassifier_optuna' instead of creating a new one.


Средний NDCG: 0.8577
Средний Precision: 0.7077
Средний Recall: 0.8258
Средний F1-Score: 0.7415
Средний NDCG: 0.8457
Средний Precision: 0.6977
Средний Recall: 0.8161
Средний F1-Score: 0.7317


[I 2026-05-02 17:55:25,007] Trial 37 finished with value: 0.8537428397310571 and parameters: {'n_estimators': 750, 'max_depth': 23, 'min_samples_split': 16, 'min_samples_leaf': 9}. Best is trial 37 with value: 0.8537428397310571.


Средний NDCG: 0.8537
Средний Precision: 0.7038
Средний Recall: 0.8209
Средний F1-Score: 0.7374
🏃 View run 37 at: http://127.0.0.1:5000/#/experiments/1/runs/1c7dd0116f3a44fa84d984e76db3a933
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8575
Средний Precision: 0.7120
Средний Recall: 0.8258
Средний F1-Score: 0.7444
Средний NDCG: 0.8459
Средний Precision: 0.7036
Средний Recall: 0.8145
Средний F1-Score: 0.7348


[I 2026-05-02 17:58:10,338] Trial 38 finished with value: 0.8536395388050088 and parameters: {'n_estimators': 750, 'max_depth': 23, 'min_samples_split': 16, 'min_samples_leaf': 8}. Best is trial 37 with value: 0.8537428397310571.


...

Средний NDCG: 0.8542
Средний Precision: 0.7198
Средний Recall: 0.8160
Средний F1-Score: 0.7457
🏃 View run 44 at: http://127.0.0.1:5000/#/experiments/1/runs/37858b71a1e843708be6ca744d74c055
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8579
Средний Precision: 0.7237
Средний Recall: 0.8205
Средний F1-Score: 0.7498
Средний NDCG: 0.8464
Средний Precision: 0.7114
Средний Recall: 0.8084
Средний F1-Score: 0.7375


[I 2026-05-02 18:20:29,713] Trial 45 finished with value: 0.8541202877959984 and parameters: {'n_estimators': 900, 'max_depth': 17, 'min_samples_split': 12, 'min_samples_leaf': 5}. Best is trial 41 with value: 0.8544867109153487.


Средний NDCG: 0.8541
Средний Precision: 0.7192
Средний Recall: 0.8155
Средний F1-Score: 0.7451
🏃 View run 45 at: http://127.0.0.1:5000/#/experiments/1/runs/100a22ab53c445be88072c8eff1377dd
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8581
Средний Precision: 0.7239
Средний Recall: 0.8204
Средний F1-Score: 0.7499
Средний NDCG: 0.8464
Средний Precision: 0.7112
Средний Recall: 0.8085
Средний F1-Score: 0.7374


[I 2026-05-02 18:23:53,249] Trial 46 finished with value: 0.854246943709631 and parameters: {'n_estimators': 950, 'max_depth': 17, 'min_samples_split': 12, 'min_samples_leaf': 5}. Best is trial 41 with value: 0.8544867109153487.


Средний NDCG: 0.8542
Средний Precision: 0.7195
Средний Recall: 0.8159
Средний F1-Score: 0.7455
🏃 View run 46 at: http://127.0.0.1:5000/#/experiments/1/runs/d17383d945e34558aecacc6b9d0bd8cd
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Number of finished trials: 47
Best params: {'n_estimators': 850, 'max_depth': 23, 'min_samples_split': 16, 'min_samples_leaf': 6}


In [57]:
study.best_trial.user_attrs['pipeline_params']

{'model__n_estimators': 850,
 'model__max_depth': 23,
 'model__min_samples_split': 16,
 'model__min_samples_leaf': 6}

In [58]:
pipeline_rf_best_optuna = Pipeline([
        ('preprocessing', ColumnTransformer([
            ('numeric_scaling', StandardScaler(), numeric_features),
            ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
        ])),
        ('model', RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced'))
    ])

pipeline_rf_best_optuna.set_params(**study.best_trial.user_attrs['pipeline_params'])
pipeline_rf_best_optuna.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric_scaling', ...), ('categorical_encoding', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of

In [59]:
y_pred_proba = pipeline_rf_best_optuna.predict_proba(X_test)

df_test = df.loc[X_test.index].copy()
df_test['y_pred_proba'] = y_pred_proba[:, 1]

In [60]:
ndcg_rf, precision_rf, recall_rf, f1_rf = calculate_metrics(df_test)

metrics_rf = {
    'NDCG': ndcg_rf,
    'Precision': precision_rf,
    'Recall': recall_rf,
    'F1': f1_rf
}

Средний NDCG: 0.7763
Средний Precision: 0.6656
Средний Recall: 0.7445
Средний F1-Score: 0.6867


In [61]:
RUN_NAME = "best_optuna_rf" 
REGISTRY_MODEL_NAME = "best_optuna_rf"

In [62]:
signature = mlflow.models.infer_signature(X_test, y_test)
input_example = X_test[:10]
code_paths = ["ML_Experiments.ipynb"]

try:
    experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
except:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id

    lr_info = mlflow.sklearn.log_model(sk_model=pipeline_rf_best_optuna, 
                                       artifact_path='best_optuna_rf',
                                       registered_model_name=REGISTRY_MODEL_NAME,
                                       input_example=input_example,
                                       code_paths=code_paths,
                                       await_registration_for=60
                                      )
    mlflow.log_metrics(metrics_rf)
    mlflow.log_params(best_params_optuna)

Registered model 'best_optuna_rf' already exists. Creating a new version of this model...
2026/05/02 18:25:29 INFO mlflow.store.model_registry.abstract_store: Waiting up to 60 seconds for model version to finish creation. Model name: best_optuna_rf, version 4


🏃 View run best_optuna_rf at: http://127.0.0.1:5000/#/experiments/1/runs/5a17184332e74ba6ac1ba9c9d3cc09d6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


Created version '4' of model 'best_optuna_rf'.


# XGBClassifier

In [63]:
def objective_xgb(trial: optuna.Trial) -> float:
    params = {
        'model__n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=50),
        'model__max_depth': trial.suggest_int('max_depth', 3, 15),
        'model__learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'model__min_child_weight': trial.suggest_int('min_child_weight', 1, 10)
    }
    
    pipeline_xgb = Pipeline([
        ('preprocessing', ColumnTransformer([
            ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
        ], remainder='passthrough')),
        ('model', XGBClassifier(
            random_state=RANDOM_STATE, 
            eval_metric='logloss', 
            use_label_encoder=False, 
            verbosity=0,
            scale_pos_weight=(len(y_train) - sum(y_train)) / sum(y_train)
        ))
    ])
    
    pipeline_xgb.set_params(**params)
    
    kfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    
    for train_idx, val_idx in kfold.split(X_train, y_train):
        X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        pipeline_xgb.fit(X_fold_train, y_fold_train)
        y_pred_proba = pipeline_xgb.predict_proba(X_fold_val)
        
        df_val = df.loc[X_fold_val.index].copy()
        df_val['y_pred_proba'] = y_pred_proba[:, 1]
        
        ndcg, _, _, _ = calculate_metrics(df_val)
        
        trial.set_user_attr('pipeline_params', params)
    
    return ndcg

In [64]:
try:
    experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
except:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)

RUN_NAME_OPTUNE_XGB = 'XGBClassifier_optuna'

with mlflow.start_run(run_name=RUN_NAME_OPTUNE_XGB, experiment_id=experiment_id) as run:
    run_id_xgb = run.info.run_id

🏃 View run XGBClassifier_optuna at: http://127.0.0.1:5000/#/experiments/1/runs/2e4226a0e5754c2bafb80f528f3894bf
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [65]:
STUDY_DB_NAME = "sqlite:///local.study.db"
STUDY_NAME_XGB = "XGBClassifier_optuna"

mlflc_xgb = MLflowCallback(
    tracking_uri=f'http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}',
    metric_name="NDCG",
    create_experiment=False,
    mlflow_kwargs={'experiment_id': experiment_id, 'tags': {MLFLOW_PARENT_RUN_ID: run_id_xgb}}
)

In [66]:
study_xgb = optuna.create_study(direction='maximize', 
                                sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
                                study_name=STUDY_NAME_XGB,
                                storage=STUDY_DB_NAME,
                                load_if_exists=True)

study_xgb.optimize(objective_xgb, 
                   n_trials=10, 
                   callbacks=[mlflc_xgb]
                  )

best_params_xgb = study_xgb.best_params
print(f"Number of finished trials: {len(study_xgb.trials)}")
print(f"Best params XGBoost: {best_params_xgb}")

[I 2026-05-02 18:25:30,028] Using an existing study with name 'XGBClassifier_optuna' instead of creating a new one.


Средний NDCG: 0.8602
Средний Precision: 0.7604
Средний Recall: 0.8154
Средний F1-Score: 0.7714
Средний NDCG: 0.8477
Средний Precision: 0.7506
Средний Recall: 0.8025
Средний F1-Score: 0.7605


[I 2026-05-02 18:25:51,610] Trial 30 finished with value: 0.856887605036557 and parameters: {'n_estimators': 850, 'max_depth': 8, 'learning_rate': 0.14174259220821753, 'min_child_weight': 10}. Best is trial 21 with value: 0.8583336583209565.


Средний NDCG: 0.8569
Средний Precision: 0.7594
Средний Recall: 0.8075
Средний F1-Score: 0.7674
🏃 View run 30 at: http://127.0.0.1:5000/#/experiments/1/runs/dfbbae0118f34cb9b7097821436a8885
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8601
Средний Precision: 0.7692
Средний Recall: 0.8056
Средний F1-Score: 0.7716
Средний NDCG: 0.8469
Средний Precision: 0.7572
Средний Recall: 0.7914
Средний F1-Score: 0.7587


[I 2026-05-02 18:26:11,981] Trial 31 finished with value: 0.8559049472166943 and parameters: {'n_estimators': 600, 'max_depth': 10, 'learning_rate': 0.21504927794657958, 'min_child_weight': 7}. Best is trial 21 with value: 0.8583336583209565.


...

Средний NDCG: 0.8571
Средний Precision: 0.7551
Средний Recall: 0.8078
Средний F1-Score: 0.7652
🏃 View run 37 at: http://127.0.0.1:5000/#/experiments/1/runs/4d2e64973b5e4c1bad6e51464b61cdf7
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8611
Средний Precision: 0.7361
Средний Recall: 0.8287
Средний F1-Score: 0.7613
Средний NDCG: 0.8484
Средний Precision: 0.7231
Средний Recall: 0.8153
Средний F1-Score: 0.7488


[I 2026-05-02 18:28:24,421] Trial 38 finished with value: 0.8564685027187541 and parameters: {'n_estimators': 400, 'max_depth': 9, 'learning_rate': 0.06958302446511004, 'min_child_weight': 10}. Best is trial 21 with value: 0.8583336583209565.


Средний NDCG: 0.8565
Средний Precision: 0.7354
Средний Recall: 0.8196
Средний F1-Score: 0.7582
🏃 View run 38 at: http://127.0.0.1:5000/#/experiments/1/runs/541b23c5be3d4a1488489031d930522e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8610
Средний Precision: 0.7289
Средний Recall: 0.8318
Средний F1-Score: 0.7585
Средний NDCG: 0.8484
Средний Precision: 0.7184
Средний Recall: 0.8204
Средний F1-Score: 0.7484


[I 2026-05-02 18:28:45,987] Trial 39 finished with value: 0.8572890201817873 and parameters: {'n_estimators': 850, 'max_depth': 8, 'learning_rate': 0.037077779749876055, 'min_child_weight': 9}. Best is trial 21 with value: 0.8583336583209565.


Средний NDCG: 0.8573
Средний Precision: 0.7300
Средний Recall: 0.8216
Средний F1-Score: 0.7555
🏃 View run 39 at: http://127.0.0.1:5000/#/experiments/1/runs/0b6a28d29c704cdd98fef1db1650d310
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Number of finished trials: 40
Best params XGBoost: {'n_estimators': 600, 'max_depth': 10, 'learning_rate': 0.09866580252678012, 'min_child_weight': 10}


In [67]:
pipeline_xgb_best = Pipeline([
    ('preprocessing', ColumnTransformer([
        ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
    ], remainder='passthrough')),
    ('model', XGBClassifier(
        random_state=RANDOM_STATE, 
        eval_metric='logloss', 
        use_label_encoder=False, 
        verbosity=0,
        scale_pos_weight=(len(y_train) - sum(y_train)) / sum(y_train)
    ))
])

pipeline_xgb_best.set_params(**study_xgb.best_trial.user_attrs['pipeline_params'])
pipeline_xgb_best.fit(X_train, y_train)

y_pred_proba_xgb = pipeline_xgb_best.predict_proba(X_test)

df_test_xgb = df.loc[X_test.index].copy()
df_test_xgb['y_pred_proba'] = y_pred_proba_xgb[:, 1]

ndcg_xgb, precision_xgb, recall_xgb, f1_xgb = calculate_metrics(df_test_xgb)
metrics_xgb = {
    'NDCG': ndcg_xgb,
    'Precision': precision_xgb,
    'Recall': recall_xgb,
    'F1': f1_xgb
}

Средний NDCG: 0.7792
Средний Precision: 0.6968
Средний Recall: 0.7411
Средний F1-Score: 0.7054


In [68]:
RUN_NAME_XGB = "best_optuna_xgb"
REGISTRY_MODEL_NAME_XGB = "best_optuna_xgb"

signature = mlflow.models.infer_signature(X_test, y_test)
input_example = X_test[:10]
code_paths = ["ML_Experiments.ipynb"]

with mlflow.start_run(run_name=RUN_NAME_XGB, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    
    xgb_info = mlflow.sklearn.log_model(sk_model=pipeline_xgb_best, 
                                       artifact_path='best_optuna_xgb',
                                       registered_model_name=REGISTRY_MODEL_NAME_XGB,
                                       input_example=input_example,
                                       code_paths=code_paths,
                                       await_registration_for=60
                                      )
    mlflow.log_metrics(metrics_xgb)
    mlflow.log_params(best_params_xgb)

Registered model 'best_optuna_xgb' already exists. Creating a new version of this model...
2026/05/02 18:28:54 INFO mlflow.store.model_registry.abstract_store: Waiting up to 60 seconds for model version to finish creation. Model name: best_optuna_xgb, version 5


🏃 View run best_optuna_xgb at: http://127.0.0.1:5000/#/experiments/1/runs/992abd1daef0404ab54769d8c8abcd84
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


Created version '5' of model 'best_optuna_xgb'.


# LGBMClassifier

In [69]:
def objective_lgbm(trial: optuna.Trial) -> float:
    params = {
        'model__n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=50),
        'model__max_depth': trial.suggest_int('max_depth', 3, 15),
        'model__learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'model__num_leaves': trial.suggest_int('num_leaves', 20, 300),
        'model__min_child_samples': trial.suggest_int('min_child_samples', 5, 100)
    }
    
    pipeline_lgbm = Pipeline([
        ('preprocessing', ColumnTransformer([
            ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
        ], remainder='passthrough')),
        ('model', LGBMClassifier(
            random_state=RANDOM_STATE, 
            verbose=-1,
            class_weight='balanced'
        ))
    ])
    
    pipeline_lgbm.set_params(**params)
    
    kfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    
    for train_idx, val_idx in kfold.split(X_train, y_train):
        X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        pipeline_lgbm.fit(X_fold_train, y_fold_train)
        y_pred_proba = pipeline_lgbm.predict_proba(X_fold_val)
        
        df_val = df.loc[X_fold_val.index].copy()
        df_val['y_pred_proba'] = y_pred_proba[:, 1]
        
        ndcg, _, _, _ = calculate_metrics(df_val)
        
        trial.set_user_attr('pipeline_params', params)
    
    return ndcg

In [70]:
RUN_NAME_OPTUNE_LGBM = 'LGBMClassifier_optuna'

with mlflow.start_run(run_name=RUN_NAME_OPTUNE_LGBM, experiment_id=experiment_id) as run:
    run_id_lgbm = run.info.run_id

🏃 View run LGBMClassifier_optuna at: http://127.0.0.1:5000/#/experiments/1/runs/6ac169d229b442d7b683d154c80f7321
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [71]:
STUDY_NAME_LGBM = "LGBMClassifier_optuna"

mlflc_lgbm = MLflowCallback(
    tracking_uri=f'http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}',
    metric_name="NDCG",
    create_experiment=False,
    mlflow_kwargs={'experiment_id': experiment_id, 'tags': {MLFLOW_PARENT_RUN_ID: run_id_lgbm}}
)

In [72]:
study_lgbm = optuna.create_study(direction='maximize', 
                                 sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
                                 study_name=STUDY_NAME_LGBM,
                                 storage=STUDY_DB_NAME,
                                 load_if_exists=True)

study_lgbm.optimize(objective_lgbm, 
                    n_trials=10, 
                    callbacks=[mlflc_lgbm]
                   )

best_params_lgbm = study_lgbm.best_params
print(f"Number of finished trials: {len(study_lgbm.trials)}")
print(f"Best params LGBM: {best_params_lgbm}")

[I 2026-05-02 18:28:54,996] Using an existing study with name 'LGBMClassifier_optuna' instead of creating a new one.


Средний NDCG: 0.8621
Средний Precision: 0.7487
Средний Recall: 0.8279
Средний F1-Score: 0.7694
Средний NDCG: 0.8496
Средний Precision: 0.7350
Средний Recall: 0.8116
Средний F1-Score: 0.7547


[I 2026-05-02 18:30:53,263] Trial 30 finished with value: 0.8579844279279389 and parameters: {'n_estimators': 850, 'max_depth': 15, 'learning_rate': 0.021865968420201474, 'num_leaves': 160, 'min_child_samples': 42}. Best is trial 21 with value: 0.8583427584888133.


Средний NDCG: 0.8580
Средний Precision: 0.7447
Средний Recall: 0.8173
Средний F1-Score: 0.7635
🏃 View run 30 at: http://127.0.0.1:5000/#/experiments/1/runs/a32fa0ec3e7b4f8299d622f42bb590b0
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8621
Средний Precision: 0.7551
Средний Recall: 0.8236
Средний F1-Score: 0.7719
Средний NDCG: 0.8487
Средний Precision: 0.7436
Средний Recall: 0.8114
Средний F1-Score: 0.7603


[I 2026-05-02 18:32:19,357] Trial 31 finished with value: 0.8575940281569361 and parameters: {'n_estimators': 950, 'max_depth': 14, 'learning_rate': 0.040429825470605835, 'num_leaves': 98, 'min_child_samples': 52}. Best is trial 21 with value: 0.8583427584888133.


...

Средний NDCG: 0.8577
Средний Precision: 0.7289
Средний Recall: 0.8241
Средний F1-Score: 0.7558
🏃 View run 37 at: http://127.0.0.1:5000/#/experiments/1/runs/541ed975c475409db19da96ff31d7220
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8621
Средний Precision: 0.7375
Средний Recall: 0.8287
Средний F1-Score: 0.7629
Средний NDCG: 0.8488
Средний Precision: 0.7297
Средний Recall: 0.8170
Средний F1-Score: 0.7536


[I 2026-05-02 18:40:51,112] Trial 38 finished with value: 0.8580797586232415 and parameters: {'n_estimators': 1000, 'max_depth': 12, 'learning_rate': 0.017051332507831654, 'num_leaves': 145, 'min_child_samples': 25}. Best is trial 21 with value: 0.8583427584888133.


Средний NDCG: 0.8581
Средний Precision: 0.7383
Средний Recall: 0.8211
Средний F1-Score: 0.7611
🏃 View run 38 at: http://127.0.0.1:5000/#/experiments/1/runs/047db15676aa46228264481560c19306
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8584
Средний Precision: 0.6884
Средний Recall: 0.8368
Средний F1-Score: 0.7340
Средний NDCG: 0.8479
Средний Precision: 0.6780
Средний Recall: 0.8266
Средний F1-Score: 0.7231


[I 2026-05-02 18:41:25,512] Trial 39 finished with value: 0.8566046782397463 and parameters: {'n_estimators': 850, 'max_depth': 14, 'learning_rate': 0.025870511876877358, 'num_leaves': 29, 'min_child_samples': 45}. Best is trial 21 with value: 0.8583427584888133.


Средний NDCG: 0.8566
Средний Precision: 0.6892
Средний Recall: 0.8341
Средний F1-Score: 0.7335
🏃 View run 39 at: http://127.0.0.1:5000/#/experiments/1/runs/14cfdf24e7c9470b8f8980107fc40bf4
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Number of finished trials: 40
Best params LGBM: {'n_estimators': 600, 'max_depth': 13, 'learning_rate': 0.07129136152747409, 'num_leaves': 57, 'min_child_samples': 57}


In [73]:
pipeline_lgbm_best = Pipeline([
    ('preprocessing', ColumnTransformer([
        ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
    ], remainder='passthrough')),
    ('model', LGBMClassifier(
        random_state=RANDOM_STATE, 
        verbose=-1,
        class_weight='balanced'
    ))
])

pipeline_lgbm_best.set_params(**study_lgbm.best_trial.user_attrs['pipeline_params'])
pipeline_lgbm_best.fit(X_train, y_train)

y_pred_proba_lgbm = pipeline_lgbm_best.predict_proba(X_test)

df_test_lgbm = df.loc[X_test.index].copy()
df_test_lgbm['y_pred_proba'] = y_pred_proba_lgbm[:, 1]

ndcg_lgbm, precision_lgbm, recall_lgbm, f1_lgbm = calculate_metrics(df_test_lgbm)
metrics_lgbm = {
    'NDCG': ndcg_lgbm,
    'Precision': precision_lgbm,
    'Recall': recall_lgbm,
    'F1': f1_lgbm
}

Средний NDCG: 0.7799
Средний Precision: 0.6742
Средний Recall: 0.7525
Средний F1-Score: 0.6957


In [74]:
RUN_NAME_LGBM = "best_optuna_lgbm"
REGISTRY_MODEL_NAME_LGBM = "best_optuna_lgbm"

signature = mlflow.models.infer_signature(X_test, y_test)
input_example = X_test[:10]
code_paths = ["ML_Experiments.ipynb"]

with mlflow.start_run(run_name=RUN_NAME_LGBM, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    
    lgbm_info = mlflow.sklearn.log_model(sk_model=pipeline_lgbm_best, 
                                        artifact_path='best_optuna_lgbm',
                                        registered_model_name=REGISTRY_MODEL_NAME_LGBM,
                                        input_example=input_example,
                                        code_paths=code_paths,
                                        await_registration_for=60
                                       )
    mlflow.log_metrics(metrics_lgbm)
    mlflow.log_params(best_params_lgbm)

Registered model 'best_optuna_lgbm' already exists. Creating a new version of this model...
2026/05/02 18:41:41 INFO mlflow.store.model_registry.abstract_store: Waiting up to 60 seconds for model version to finish creation. Model name: best_optuna_lgbm, version 5


🏃 View run best_optuna_lgbm at: http://127.0.0.1:5000/#/experiments/1/runs/db648fe2e7954d439aa4861f1cb1f47b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


Created version '5' of model 'best_optuna_lgbm'.


# CatBoostClassifier

In [75]:
def objective_catboost(trial: optuna.Trial) -> float:
    params = {
        'model__iterations': trial.suggest_int('iterations', 100, 1000, step=50),
        'model__depth': trial.suggest_int('depth', 3, 10),
        'model__learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'model__l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10, log=True)
    }
    
    pipeline_catboost = Pipeline([
        ('preprocessing', ColumnTransformer([
            ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
        ], remainder='passthrough')),
        ('model', CatBoostClassifier(
            random_state=RANDOM_STATE, 
            verbose=0, 
            auto_class_weights='Balanced'
        ))
    ])
    
    pipeline_catboost.set_params(**params)
    
    kfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    
    for train_idx, val_idx in kfold.split(X_train, y_train):
        X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        pipeline_catboost.fit(X_fold_train, y_fold_train)
        y_pred_proba = pipeline_catboost.predict_proba(X_fold_val)
        
        df_val = df.loc[X_fold_val.index].copy()
        df_val['y_pred_proba'] = y_pred_proba[:, 1]
        
        ndcg, _, _, _ = calculate_metrics(df_val)
        
        trial.set_user_attr('pipeline_params', params)
    
    return ndcg

In [76]:
RUN_NAME_OPTUNE_CATBOOST = 'CatBoostClassifier_optuna'

with mlflow.start_run(run_name=RUN_NAME_OPTUNE_CATBOOST, experiment_id=experiment_id) as run:
    run_id_catboost = run.info.run_id

🏃 View run CatBoostClassifier_optuna at: http://127.0.0.1:5000/#/experiments/1/runs/33bf86a3281d42219e3bf130eeac98f7
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [77]:
STUDY_NAME_CATBOOST = "CatBoostClassifier_optuna"

mlflc_catboost = MLflowCallback(
    tracking_uri=f'http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}',
    metric_name="NDCG",
    create_experiment=False,
    mlflow_kwargs={'experiment_id': experiment_id, 'tags': {MLFLOW_PARENT_RUN_ID: run_id_catboost}}
)

In [78]:
study_catboost = optuna.create_study(direction='maximize', 
                                     sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
                                     study_name=STUDY_NAME_CATBOOST,
                                     storage=STUDY_DB_NAME,
                                     load_if_exists=True)

study_catboost.optimize(objective_catboost, 
                        n_trials=10, 
                        callbacks=[mlflc_catboost]
                       )

best_params_catboost = study_catboost.best_params
print(f"Number of finished trials: {len(study_catboost.trials)}")
print(f"Best params CatBoost: {best_params_catboost}")

[I 2026-05-02 18:41:41,485] Using an existing study with name 'CatBoostClassifier_optuna' instead of creating a new one.


Средний NDCG: 0.8568
Средний Precision: 0.6849
Средний Recall: 0.8361
Средний F1-Score: 0.7311
Средний NDCG: 0.8459
Средний Precision: 0.6718
Средний Recall: 0.8250
Средний F1-Score: 0.7175


[I 2026-05-02 18:42:12,367] Trial 31 finished with value: 0.8534406420437601 and parameters: {'iterations': 550, 'depth': 10, 'learning_rate': 0.021865968420201474, 'l2_leaf_reg': 8.556385477088213}. Best is trial 19 with value: 0.8575655602470745.


Средний NDCG: 0.8534
Средний Precision: 0.6838
Средний Recall: 0.8297
Средний F1-Score: 0.7280
🏃 View run 31 at: http://127.0.0.1:5000/#/experiments/1/runs/0e09d4f1c29c4d7eae4bf93bc6066777
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8616
Средний Precision: 0.7388
Средний Recall: 0.8300
Средний F1-Score: 0.7647
Средний NDCG: 0.8492
Средний Precision: 0.7304
Средний Recall: 0.8154
Средний F1-Score: 0.7534


[I 2026-05-02 18:42:44,513] Trial 32 finished with value: 0.8574832738333216 and parameters: {'iterations': 900, 'depth': 9, 'learning_rate': 0.0935313835540852, 'l2_leaf_reg': 8.30970459531273}. Best is trial 19 with value: 0.8575655602470745.


...

Средний NDCG: 0.8563
Средний Precision: 0.7510
Средний Recall: 0.8124
Средний F1-Score: 0.7652
🏃 View run 38 at: http://127.0.0.1:5000/#/experiments/1/runs/2b2c99a5224542a384b5c0d3e7da8353
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8615
Средний Precision: 0.7282
Средний Recall: 0.8315
Средний F1-Score: 0.7588
Средний NDCG: 0.8487
Средний Precision: 0.7186
Средний Recall: 0.8198
Средний F1-Score: 0.7478


[I 2026-05-02 18:46:22,046] Trial 39 finished with value: 0.8571619169936802 and parameters: {'iterations': 850, 'depth': 8, 'learning_rate': 0.10389590734703241, 'l2_leaf_reg': 9.600866232623389}. Best is trial 36 with value: 0.8577557554231894.


Средний NDCG: 0.8572
Средний Precision: 0.7290
Средний Recall: 0.8251
Средний F1-Score: 0.7564
🏃 View run 39 at: http://127.0.0.1:5000/#/experiments/1/runs/326980af9c434dceaf59110923dda050
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8606
Средний Precision: 0.7271
Средний Recall: 0.8310
Средний F1-Score: 0.7573
Средний NDCG: 0.8479
Средний Precision: 0.7215
Средний Recall: 0.8191
Средний F1-Score: 0.7497


[I 2026-05-02 18:46:44,508] Trial 40 finished with value: 0.8570947963117044 and parameters: {'iterations': 650, 'depth': 7, 'learning_rate': 0.22361544567965294, 'l2_leaf_reg': 7.370221056643942}. Best is trial 36 with value: 0.8577557554231894.


Средний NDCG: 0.8571
Средний Precision: 0.7293
Средний Recall: 0.8245
Средний F1-Score: 0.7566
🏃 View run 40 at: http://127.0.0.1:5000/#/experiments/1/runs/eae822f0c3ad4e9592220f9a45a0e749
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Number of finished trials: 41
Best params CatBoost: {'iterations': 900, 'depth': 8, 'learning_rate': 0.13266287634042745, 'l2_leaf_reg': 4.751361322705928}


In [79]:
pipeline_catboost_best = Pipeline([
    ('preprocessing', ColumnTransformer([
        ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
    ], remainder='passthrough')),
    ('model', CatBoostClassifier(
        random_state=RANDOM_STATE, 
        verbose=0, 
        auto_class_weights='Balanced'
    ))
])

pipeline_catboost_best.set_params(**study_catboost.best_trial.user_attrs['pipeline_params'])
pipeline_catboost_best.fit(X_train, y_train)

y_pred_proba_catboost = pipeline_catboost_best.predict_proba(X_test)

df_test_catboost = df.loc[X_test.index].copy()
df_test_catboost['y_pred_proba'] = y_pred_proba_catboost[:, 1]

ndcg_catboost, precision_catboost, recall_catboost, f1_catboost = calculate_metrics(df_test_catboost)
metrics_catboost = {
    'NDCG': ndcg_catboost,
    'Precision': precision_catboost,
    'Recall': recall_catboost,
    'F1': f1_catboost
}

Средний NDCG: 0.7792
Средний Precision: 0.6787
Средний Recall: 0.7497
Средний F1-Score: 0.6975


In [80]:
RUN_NAME_CATBOOST = "best_optuna_catboost"
REGISTRY_MODEL_NAME_CATBOOST = "best_optuna_catboost"

signature = mlflow.models.infer_signature(X_test, y_test)
input_example = X_test[:10]
code_paths = ["ML_Experiments.ipynb"]

with mlflow.start_run(run_name=RUN_NAME_CATBOOST, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    
    catboost_info = mlflow.sklearn.log_model(sk_model=pipeline_catboost_best, 
                                             artifact_path='best_optuna_catboost',
                                             registered_model_name=REGISTRY_MODEL_NAME_CATBOOST,
                                             input_example=input_example,
                                             code_paths=code_paths,
                                             await_registration_for=60
                                            )
    mlflow.log_metrics(metrics_catboost)
    mlflow.log_params(best_params_catboost)

Registered model 'best_optuna_catboost' already exists. Creating a new version of this model...
2026/05/02 18:46:57 INFO mlflow.store.model_registry.abstract_store: Waiting up to 60 seconds for model version to finish creation. Model name: best_optuna_catboost, version 6


🏃 View run best_optuna_catboost at: http://127.0.0.1:5000/#/experiments/1/runs/a7498421afa349bcb927b93142e63336
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


Created version '6' of model 'best_optuna_catboost'.


# Model comparison

In [81]:
models_comparison = pd.DataFrame({
    'Model': ['LogisticRegression', 'RandomForest', 'XGBoost', 'LightGBM', 'CatBoost'],
    'NDCG': [ndcg_lr, ndcg_rf, 
             ndcg_xgb, ndcg_lgbm, ndcg_catboost],
    'Precision': [precision_lr, precision_rf, 
                  precision_xgb, precision_lgbm, precision_catboost],
    'Recall': [recall_lr, recall_rf, 
               recall_xgb, recall_lgbm, recall_catboost],
    'F1': [f1_lr, f1_rf, 
           f1_xgb, f1_lgbm, f1_catboost]
})

models_comparison

,Model,NDCG,Precision,Recall,F1
0,LogisticRegression,0.751483,0.638633,0.598481,0.602124
1,RandomForest,0.776260,0.665618,0.744510,0.686675
2,XGBoost,0.779192,0.696847,0.741136,0.705449
3,LightGBM,0.779915,0.674173,0.752533,0.695656
4,CatBoost,0.779173,0.678703,0.749736,0.697520


In [82]:
best_model_idx = models_comparison['NDCG'].idxmax()
print(f"\nЛучшая модель: {models_comparison.iloc[best_model_idx]['Model']} с NDCG = {models_comparison.iloc[best_model_idx]['NDCG']:.4f}")


Лучшая модель: LightGBM с NDCG = 0.7799


<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

По итогам подбора гиперпараметров лучшее качество показала модель `XGBClassifier`. Теперь попробуем добавить дополнительные признаки от кандидата генераторов. В первую очередь используем коллаборативную фильтрацию.

</div>

# ALS

## Feature engineering

In [83]:
def create_interaction_matrix(df):
    unique_vacancies = df['vacancy_id'].unique().tolist()
    unique_resumes = df['resume_id'].unique().tolist()

    id2vacancy = dict(enumerate(unique_vacancies))
    id2resume = dict(enumerate(unique_resumes))
    vacancy2id = {v: k for k, v in id2vacancy.items()}
    resume2id = {v: k for k, v in id2resume.items()}

    rows = [vacancy2id[vacancy] for vacancy in df['vacancy_id']]
    cols = [resume2id[resume] for resume in df['resume_id']]

    interactions_matrix = csr_matrix(
        (df['target'], (rows, cols)),
        shape=(len(unique_vacancies), len(unique_resumes)),
        dtype=np.float32,
    )

    return interactions_matrix, id2vacancy, id2resume, vacancy2id, resume2id, unique_vacancies, unique_resumes

In [84]:
interactions_matrix, id2vacancy, id2resume, vacancy2id, resume2id, unique_vacancies, unique_resumes = create_interaction_matrix(df)

print(f"Размерность матрицы взаимодействий: {interactions_matrix.shape}")
print(f"Плотность матрицы: {interactions_matrix.nnz / (interactions_matrix.shape[0] * interactions_matrix.shape[1]):.6f}")

Размерность матрицы взаимодействий: (3409, 20130)
Плотность матрицы: 0.004687


In [85]:
def get_factors(interactions_matrix):
    als_model = AlternatingLeastSquares(
        factors=64,          
        regularization=0.1,  
        iterations=30,       
        random_state=RANDOM_STATE,
        num_threads=0
    )
    
    als_model.fit(interactions_matrix.T)

    vacancy_factors = als_model.item_factors
    resume_factors = als_model.user_factors

    return vacancy_factors, resume_factors

In [86]:
vacancy_factors, resume_factors = get_factors(interactions_matrix)

print(f"Размерность эмбеддингов вакансий: {vacancy_factors.shape}")
print(f"Размерность эмбеддингов резюме: {resume_factors.shape}")

  0%|          | 0/30 [00:00<?, ?it/s]

Размерность эмбеддингов вакансий: (3409, 64)
Размерность эмбеддингов резюме: (20130, 64)


In [87]:
def get_als_score(vacancy_id, resume_id):
    if vacancy_id not in vacancy2id or resume_id not in resume2id:
        return 0
    else:
        vac_idx = vacancy2id[vacancy_id]
        res_idx = resume2id[resume_id]
        score = np.dot(vacancy_factors[vac_idx], resume_factors[res_idx])
        return score

df['als_score'] = df.apply(
    lambda row: get_als_score(row['vacancy_id'], row['resume_id']), 
    axis=1
)

In [88]:
df[['vacancy_id', 'resume_id', 'target', 'als_score']].head()

,vacancy_id,resume_id,target,als_score
0,126167948,6969174,1,0.000042
1,126167948,9100077,1,0.000033
2,126167948,32644957,1,0.000019
3,126167948,27220466,1,0.000021
4,126167948,7532708,1,0.000021


<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

Проверим схожесть резюме по латентным векторам.

</div>

In [89]:
def find_similar_resumes(resume_id, resume_factors, n_similar=10, metric='cosine'):
    """
    Находит N наиболее похожих резюме на заданное.
    """
    if resume_id not in resume2id:
        print(f"Резюме с ID {resume_id} не найдено")
        return []

    target_idx = resume2id[resume_id]
    target_vector = resume_factors[target_idx].reshape(1, -1)
    
    if metric == 'cosine':
        similarities = cosine_similarity(target_vector, resume_factors)[0]
    else:
        similarities = np.dot(resume_factors, target_vector.T).flatten()
    
    similar_indices = np.argsort(similarities)[::-1][1:n_similar+1]
    
    similar_resumes = []
    for idx in similar_indices:
        sim_resume_id = unique_resumes[idx]
        similarity_score = similarities[idx]
        similar_resumes.append((sim_resume_id, similarity_score))
    
    return similar_resumes

def get_resume_profile(resume_id, df):
    """Вспомогательная функция для получения информации о резюме"""
    resume_data = df[df['resume_id'] == resume_id].iloc[0]
    return {
        'resume_id': resume_id,
        'title': resume_data.get('resume_title', 'N/A'),
        'specialization': resume_data.get('resume_specialization', 'N/A'),
        'skills': resume_data.get('resume_skills', 'N/A'),
        'experience': resume_data.get('resume_total_experience', 'N/A'),
        'salary': resume_data.get('resume_salary', 'N/A'),
        'location': resume_data.get('resume_location', 'N/A')
    }

In [90]:
example_resume_id = df['resume_id'].sample(1).values[0].item()
print(f"Исходное резюме (ID: {example_resume_id}):")
original_profile = get_resume_profile(example_resume_id, df)
for key, value in original_profile.items():
    print(f"  {key}: {value}")

print(f"\nТоп-5 похожих резюме (косинусная близость):")
similar_resumes = find_similar_resumes(example_resume_id, resume_factors, n_similar=5, metric='cosine')

for i, (sim_id, score) in enumerate(similar_resumes, 1):
    profile = get_resume_profile(sim_id, df)
    print(f"\n{i}. Резюме ID: {sim_id} (сходство: {score:.4f})")
    print(f"   Должность: {profile['title']}")
    print(f"   Специализация: {profile['specialization']}")
    print(f"   Локация: {profile['location']}")
    print(f"   Опыт: {profile['experience']}")

Исходное резюме (ID: 65353364):
  resume_id: 65353364
  title: Системный администратор
  specialization: ['Сетевой инженер', 'Системный администратор', 'Системный инженер']
  skills: [' Linux', 'Основы анализа данных.', 'Основы Python', 'Информационные технологии', 'Администрирование сетевого оборудования', 'Сетевые технологии', 'Active Directory', 'VLAN', 'OSPF', 'Системное администрирование', 'Техническая поддержка', 'Cisco', 'Qtech', 'Huawei', 'Eltex', 'Основы SQL', 'Helpdesk', 'Cloud', 'OSI', 'Администрирование серверов Linux', 'Zabbix']
  experience: 15 лет 10 месяцев
  salary: 170000.0
  location: Москва

Топ-5 похожих резюме (косинусная близость):

1. Резюме ID: 55976730 (сходство: 0.0000)
   Должность: Инженер-программист .Net/C#
   Специализация: ['Программист, разработчик']
   Локация: Москва
   Опыт: 5 лет 9 месяцев

2. Резюме ID: 37812953 (сходство: 0.0000)
   Должность: Joomla Developer
   Специализация: ['Программист, разработчик']
   Локация: Донецк
   Опыт: 12 лет 7 мес

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

Разделим тренировочную выборку еще на две, чтобы избежать подглядывания и добавить в признак `als_score` нулевые значения в тренировочный датасет, т.к. у нас могут быть вакансии и резюме без взаимодействий до обучения.

</div>

In [91]:
X_train_als, X_test_als, y_train_als, y_test_als = train_test_split(X_train, y_train, test_size=0.3, random_state=RANDOM_STATE, stratify=y_train)

In [92]:
train_als_interactions = df.loc[X_train_als.index, ['vacancy_id', 'resume_id', 'target']]

interactions_matrix, id2vacancy, id2resume, vacancy2id, resume2id, unique_vacancies, unique_resumes = create_interaction_matrix(train_als_interactions)
vacancy_factors, resume_factors = get_factors(interactions_matrix)

# возвращаем айдишники вакансий и резюме для расчета als score
X_train[['vacancy_id', 'resume_id']] = df.loc[X_train.index, ['vacancy_id', 'resume_id']]
X_train['als_score'] = X_train.apply(
    lambda row: get_als_score(row['vacancy_id'], row['resume_id']), 
    axis=1
)

X_train = X_train.drop(['vacancy_id', 'resume_id'], axis=1)

  0%|          | 0/30 [00:00<?, ?it/s]

In [93]:
# проверим есть ли вакансии и резюме без взаимодействий до обучения
X_train[X_train['als_score'] == 0]

,vacancy_area,vacancy_experience,vacancy_employment,vacancy_schedule,resume_salary,resume_age,resume_experience_months,resume_location,resume_gender,resume_applicant_status,resume_last_company_experience_months,location_matching,resume_skill_count_in_vacancy,last_position_in_vacancy,similarity_score_tfidf,als_score
286115,Москва,От 3 до 6 лет,Полная занятость,Удаленная работа,40000.0,38.0,223.0,Москва,Неизвестно,NDT,208.0,1,0,0.25,0.015081,0.0
216165,Москва,От 1 года до 3 лет,Полная занятость,Удаленная работа,150000.0,35.0,152.0,Москва,Мужчина,Активно ищет работу,89.0,1,0,0.00,0.038536,0.0
297591,Москва,От 3 до 6 лет,Полная занятость,Удаленная работа,150000.0,35.0,152.0,Москва,Мужчина,Активно ищет работу,89.0,1,0,0.00,0.016496,0.0
192099,Москва,Более 6 лет,Полная занятость,Полный день,30000.0,57.0,289.0,Москва,Мужчина,NDT,170.0,1,0,0.00,0.000000,0.0
82523,Москва,От 1 года до 3 лет,Полная занятость,Удаленная работа,0.0,38.0,185.0,Москва,Мужчина,NDT,114.0,1,4,0.00,0.063418,0.0
32590,Москва,От 3 до 6 лет,Полная занятость,Удаленная работа,0.0,38.0,185.0,Москва,Мужчина,NDT,114.0,1,2,0.00,0.101719,0.0
260861,Москва,От 1 года до 3 лет,Полная занятость,Удаленная работа,30000.0,57.0,289.0,Москва,Мужчина,NDT,170.0,1,0,0.00,0.020658,0.0
52865,Москва,От 3 до 6 лет,Полная занятость,Удаленная работа,150000.0,35.0,152.0,Москва,Мужчина,Активно ищет работу,89.0,1,0,0.00,0.011811,0.0
175258,Москва,От 1 года до 3 лет,Полная занятость,Удаленная работа,150000.0,35.0,152.0,Москва,Мужчина,Активно ищет работу,89.0,1,0,0.00,0.021698,0.0
217714,Москва,От 3 до 6 лет,Полная занятость,Удаленная работа,150000.0,35.0,152.0,Москва,Мужчина,Активно ищет работу,89.0,1,0,0.00,0.012118,0.0


<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

Теперь добавим признак `als_score` для тестовой выборке, но уже возьмем матрицу интеракций на полной тренировочной выборке.

</div>

In [94]:
train_interactions = df.loc[X_train.index, ['vacancy_id', 'resume_id', 'target']]

interactions_matrix, id2vacancy, id2resume, vacancy2id, resume2id, unique_vacancies, unique_resumes = create_interaction_matrix(train_interactions)
vacancy_factors, resume_factors = get_factors(interactions_matrix)

# возвращаем айдишники вакансий и резюме для расчета als score
X_test[['vacancy_id', 'resume_id']] = df.loc[X_test.index, ['vacancy_id', 'resume_id']]
X_test['als_score'] = X_test.apply(
    lambda row: get_als_score(row['vacancy_id'], row['resume_id']), 
    axis=1
)

X_test = X_test.drop(['vacancy_id', 'resume_id'], axis=1)

  0%|          | 0/30 [00:00<?, ?it/s]

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

Выборки получены, обучим `XGBClassifier` с подбором гиперпараметров.

</div>

## XGBClassifier

In [95]:
def objective_xgb(trial: optuna.Trial) -> float:
    params = {
        'model__n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=50),
        'model__max_depth': trial.suggest_int('max_depth', 3, 15),
        'model__learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'model__min_child_weight': trial.suggest_int('min_child_weight', 1, 10)
    }
    
    pipeline_xgb = Pipeline([
        ('preprocessing', ColumnTransformer([
            ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
        ], remainder='passthrough')),
        ('model', XGBClassifier(
            random_state=RANDOM_STATE, 
            eval_metric='logloss', 
            use_label_encoder=False, 
            verbosity=0,
            scale_pos_weight=(len(y_train) - sum(y_train)) / sum(y_train)
        ))
    ])
    
    pipeline_xgb.set_params(**params)
    
    kfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    
    for train_idx, val_idx in kfold.split(X_train, y_train):
        X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        pipeline_xgb.fit(X_fold_train, y_fold_train)
        y_pred_proba = pipeline_xgb.predict_proba(X_fold_val)
        
        df_val = df.loc[X_fold_val.index].copy()
        df_val['y_pred_proba'] = y_pred_proba[:, 1]
        
        ndcg, _, _, _ = calculate_metrics(df_val)
        
        trial.set_user_attr('pipeline_params', params)
    
    return ndcg

In [96]:
try:
    experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
except:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)

RUN_NAME_OPTUNE_XGB = 'XGBClassifier_optuna_als'

with mlflow.start_run(run_name=RUN_NAME_OPTUNE_XGB, experiment_id=experiment_id) as run:
    run_id_xgb = run.info.run_id

🏃 View run XGBClassifier_optuna_als at: http://127.0.0.1:5000/#/experiments/1/runs/e768f2d647a44154b35edb6f80c7a005
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [97]:
STUDY_DB_NAME = "sqlite:///local.study.db"
STUDY_NAME_XGB = "XGBClassifier_optuna_als"

mlflc_xgb = MLflowCallback(
    tracking_uri=f'http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}',
    metric_name="NDCG",
    create_experiment=False,
    mlflow_kwargs={'experiment_id': experiment_id, 'tags': {MLFLOW_PARENT_RUN_ID: run_id_xgb}}
)

In [98]:
study_xgb = optuna.create_study(direction='maximize', 
                                sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
                                study_name=STUDY_NAME_XGB,
                                storage=STUDY_DB_NAME,
                                load_if_exists=True)

study_xgb.optimize(objective_xgb, 
                   n_trials=10, 
                   callbacks=[mlflc_xgb]
                  )

best_params_xgb = study_xgb.best_params
print(f"Number of finished trials: {len(study_xgb.trials)}")
print(f"Best params XGBoost: {best_params_xgb}")

[I 2026-05-02 18:47:06,464] Using an existing study with name 'XGBClassifier_optuna_als' instead of creating a new one.


Средний NDCG: 0.8676
Средний Precision: 0.7946
Средний Recall: 0.8304
Средний F1-Score: 0.8023
Средний NDCG: 0.8575
Средний Precision: 0.7896
Средний Recall: 0.8256
Средний F1-Score: 0.7970


[I 2026-05-02 18:47:30,663] Trial 30 finished with value: 0.8626753506022636 and parameters: {'n_estimators': 750, 'max_depth': 12, 'learning_rate': 0.023336676306434698, 'min_child_weight': 7}. Best is trial 26 with value: 0.8635372074877529.


Средний NDCG: 0.8627
Средний Precision: 0.7900
Средний Recall: 0.8278
Средний F1-Score: 0.7981
🏃 View run 30 at: http://127.0.0.1:5000/#/experiments/1/runs/b5b5d403d28a4732aeae8b9e8c001411
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8677
Средний Precision: 0.7916
Средний Recall: 0.8362
Средний F1-Score: 0.8029
Средний NDCG: 0.8576
Средний Precision: 0.7840
Средний Recall: 0.8290
Средний F1-Score: 0.7955


[I 2026-05-02 18:47:52,736] Trial 31 finished with value: 0.8631615701716232 and parameters: {'n_estimators': 850, 'max_depth': 8, 'learning_rate': 0.03720002603911324, 'min_child_weight': 4}. Best is trial 26 with value: 0.8635372074877529.


...

Средний NDCG: 0.8632
Средний Precision: 0.7992
Средний Recall: 0.8222
Средний F1-Score: 0.8005
🏃 View run 37 at: http://127.0.0.1:5000/#/experiments/1/runs/103c8694e05d45bb8d1778bbd64a74da
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8676
Средний Precision: 0.7934
Средний Recall: 0.8337
Средний F1-Score: 0.8030
Средний NDCG: 0.8576
Средний Precision: 0.7880
Средний Recall: 0.8282
Средний F1-Score: 0.7974


[I 2026-05-02 18:50:21,356] Trial 38 finished with value: 0.8631761517312011 and parameters: {'n_estimators': 600, 'max_depth': 12, 'learning_rate': 0.02399111248214035, 'min_child_weight': 7}. Best is trial 32 with value: 0.8638529990938331.


Средний NDCG: 0.8632
Средний Precision: 0.7885
Средний Recall: 0.8300
Средний F1-Score: 0.7981
🏃 View run 38 at: http://127.0.0.1:5000/#/experiments/1/runs/0a5cd2fc3c74455ab019897a6ded5375
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8671
Средний Precision: 0.8017
Средний Recall: 0.8276
Средний F1-Score: 0.8050
Средний NDCG: 0.8576
Средний Precision: 0.7939
Средний Recall: 0.8202
Средний F1-Score: 0.7966


[I 2026-05-02 18:50:43,031] Trial 39 finished with value: 0.8632604190305846 and parameters: {'n_estimators': 700, 'max_depth': 10, 'learning_rate': 0.07316906365943404, 'min_child_weight': 6}. Best is trial 32 with value: 0.8638529990938331.


Средний NDCG: 0.8633
Средний Precision: 0.7999
Средний Recall: 0.8244
Средний F1-Score: 0.8020
🏃 View run 39 at: http://127.0.0.1:5000/#/experiments/1/runs/affee986cfcd41c6a442c26fa0dd3312
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Number of finished trials: 40
Best params XGBoost: {'n_estimators': 600, 'max_depth': 9, 'learning_rate': 0.0482381453391051, 'min_child_weight': 5}


In [99]:
pipeline_xgb_best = Pipeline([
    ('preprocessing', ColumnTransformer([
        ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
    ], remainder='passthrough')),
    ('model', XGBClassifier(
        random_state=RANDOM_STATE, 
        eval_metric='logloss', 
        use_label_encoder=False, 
        verbosity=0,
        scale_pos_weight=(len(y_train) - sum(y_train)) / sum(y_train)
    ))
])

pipeline_xgb_best.set_params(**study_xgb.best_trial.user_attrs['pipeline_params'])
pipeline_xgb_best.fit(X_train, y_train)

y_pred_proba_xgb = pipeline_xgb_best.predict_proba(X_test)

df_test_xgb = df.loc[X_test.index].copy()
df_test_xgb['y_pred_proba'] = y_pred_proba_xgb[:, 1]

ndcg_xgb_als, precision_xgb_als, recall_xgb_als, f1_xgb_als = calculate_metrics(df_test_xgb)
metrics_xgb_als = {
    'NDCG': ndcg_xgb_als,
    'Precision': precision_xgb_als,
    'Recall': recall_xgb_als,
    'F1': f1_xgb_als
}

Средний NDCG: 0.7812
Средний Precision: 0.7086
Средний Recall: 0.7425
Средний F1-Score: 0.7137


In [100]:
RUN_NAME_XGB = "best_optuna_xgb_als"
REGISTRY_MODEL_NAME_XGB = "best_optuna_xgb_als"

signature = mlflow.models.infer_signature(X_test, y_test)
input_example = X_test[:10]
code_paths = ["ML_Experiments.ipynb"]

with mlflow.start_run(run_name=RUN_NAME_XGB, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    
    xgb_info = mlflow.sklearn.log_model(sk_model=pipeline_xgb_best, 
                                       artifact_path='best_optuna_xgb_als',
                                       registered_model_name=REGISTRY_MODEL_NAME_XGB,
                                       input_example=input_example,
                                       code_paths=code_paths,
                                       await_registration_for=60
                                      )
    mlflow.log_metrics(metrics_xgb_als)
    mlflow.log_params(best_params_xgb)

Registered model 'best_optuna_xgb_als' already exists. Creating a new version of this model...
2026/05/02 18:50:52 INFO mlflow.store.model_registry.abstract_store: Waiting up to 60 seconds for model version to finish creation. Model name: best_optuna_xgb_als, version 4


🏃 View run best_optuna_xgb_als at: http://127.0.0.1:5000/#/experiments/1/runs/6cb0174b1509444e97ce82b1721c32e3
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


Created version '4' of model 'best_optuna_xgb_als'.


## LGBMClassifier

In [101]:
def objective_lgbm(trial: optuna.Trial) -> float:
    params = {
        'model__n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=50),
        'model__max_depth': trial.suggest_int('max_depth', 3, 15),
        'model__learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'model__num_leaves': trial.suggest_int('num_leaves', 20, 300),
        'model__min_child_samples': trial.suggest_int('min_child_samples', 5, 100)
    }
    
    pipeline_lgbm = Pipeline([
        ('preprocessing', ColumnTransformer([
            ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
        ], remainder='passthrough')),
        ('model', LGBMClassifier(
            random_state=RANDOM_STATE, 
            verbose=-1,
            class_weight='balanced'
        ))
    ])
    
    pipeline_lgbm.set_params(**params)
    
    kfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    
    for train_idx, val_idx in kfold.split(X_train, y_train):
        X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        pipeline_lgbm.fit(X_fold_train, y_fold_train)
        y_pred_proba = pipeline_lgbm.predict_proba(X_fold_val)
        
        df_val = df.loc[X_fold_val.index].copy()
        df_val['y_pred_proba'] = y_pred_proba[:, 1]
        
        ndcg, _, _, _ = calculate_metrics(df_val)
        
        trial.set_user_attr('pipeline_params', params)
    
    return ndcg

In [102]:
RUN_NAME_OPTUNE_LGBM = 'LGBMClassifier_optuna_als'

with mlflow.start_run(run_name=RUN_NAME_OPTUNE_LGBM, experiment_id=experiment_id) as run:
    run_id_lgbm = run.info.run_id

🏃 View run LGBMClassifier_optuna_als at: http://127.0.0.1:5000/#/experiments/1/runs/caf2844e60bb43edb115348df606a773
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [103]:
STUDY_NAME_LGBM = "LGBMClassifier_optuna_als"

mlflc_lgbm = MLflowCallback(
    tracking_uri=f'http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}',
    metric_name="NDCG",
    create_experiment=False,
    mlflow_kwargs={'experiment_id': experiment_id, 'tags': {MLFLOW_PARENT_RUN_ID: run_id_lgbm}}
)

In [104]:
study_lgbm = optuna.create_study(direction='maximize', 
                                 sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
                                 study_name=STUDY_NAME_LGBM,
                                 storage=STUDY_DB_NAME,
                                 load_if_exists=True)

study_lgbm.optimize(objective_lgbm, 
                    n_trials=10, 
                    callbacks=[mlflc_lgbm]
                   )

best_params_lgbm = study_lgbm.best_params
print(f"Number of finished trials: {len(study_lgbm.trials)}")
print(f"Best params LGBM: {best_params_lgbm}")

[I 2026-05-02 18:50:52,153] Using an existing study with name 'LGBMClassifier_optuna_als' instead of creating a new one.


Средний NDCG: 0.8677
Средний Precision: 0.8063
Средний Recall: 0.8156
Средний F1-Score: 0.8016
Средний NDCG: 0.8566
Средний Precision: 0.8021
Средний Recall: 0.8111
Средний F1-Score: 0.7971


[I 2026-05-02 18:52:10,569] Trial 30 finished with value: 0.8630141923640238 and parameters: {'n_estimators': 900, 'max_depth': 12, 'learning_rate': 0.14849387713097773, 'num_leaves': 95, 'min_child_samples': 58}. Best is trial 19 with value: 0.8642042129240123.


Средний NDCG: 0.8630
Средний Precision: 0.8055
Средний Recall: 0.8106
Средний F1-Score: 0.7984
🏃 View run 30 at: http://127.0.0.1:5000/#/experiments/1/runs/b251d18e72cb499682e6919570bd7948
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8683
Средний Precision: 0.7688
Средний Recall: 0.8507
Средний F1-Score: 0.7949
Средний NDCG: 0.8581
Средний Precision: 0.7592
Средний Recall: 0.8405
Средний F1-Score: 0.7851


[I 2026-05-02 18:52:43,108] Trial 31 finished with value: 0.8637209804292143 and parameters: {'n_estimators': 950, 'max_depth': 11, 'learning_rate': 0.030241001410559513, 'num_leaves': 23, 'min_child_samples': 50}. Best is trial 19 with value: 0.8642042129240123.


...

Средний NDCG: 0.8630
Средний Precision: 0.7920
Средний Recall: 0.8324
Средний F1-Score: 0.8010
🏃 View run 37 at: http://127.0.0.1:5000/#/experiments/1/runs/bb6addc19e0942d4ad1f7e54de952bef
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8683
Средний Precision: 0.7881
Средний Recall: 0.8427
Средний F1-Score: 0.8035
Средний NDCG: 0.8579
Средний Precision: 0.7760
Средний Recall: 0.8331
Средний F1-Score: 0.7922


[I 2026-05-02 18:59:34,390] Trial 38 finished with value: 0.8631129971735638 and parameters: {'n_estimators': 950, 'max_depth': 11, 'learning_rate': 0.023544850060403456, 'num_leaves': 66, 'min_child_samples': 44}. Best is trial 19 with value: 0.8642042129240123.


Средний NDCG: 0.8631
Средний Precision: 0.7844
Средний Recall: 0.8378
Средний F1-Score: 0.7990
🏃 View run 38 at: http://127.0.0.1:5000/#/experiments/1/runs/d72bcf9af24f442a9aa3a2d46bc733f7
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8673
Средний Precision: 0.8059
Средний Recall: 0.8242
Средний F1-Score: 0.8056
Средний NDCG: 0.8578
Средний Precision: 0.7999
Средний Recall: 0.8215
Средний F1-Score: 0.8009


[I 2026-05-02 19:00:56,985] Trial 39 finished with value: 0.8635360714693456 and parameters: {'n_estimators': 700, 'max_depth': 14, 'learning_rate': 0.061001355091202455, 'num_leaves': 129, 'min_child_samples': 35}. Best is trial 19 with value: 0.8642042129240123.


Средний NDCG: 0.8635
Средний Precision: 0.8020
Средний Recall: 0.8186
Средний F1-Score: 0.8002
🏃 View run 39 at: http://127.0.0.1:5000/#/experiments/1/runs/87c5c777694d4d06a68ca11acb88b3a9
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Number of finished trials: 40
Best params LGBM: {'n_estimators': 900, 'max_depth': 13, 'learning_rate': 0.04019165012583699, 'num_leaves': 23, 'min_child_samples': 54}


In [105]:
pipeline_lgbm_best = Pipeline([
    ('preprocessing', ColumnTransformer([
        ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
    ], remainder='passthrough')),
    ('model', LGBMClassifier(
        random_state=RANDOM_STATE, 
        verbose=-1,
        class_weight='balanced'
    ))
])

pipeline_lgbm_best.set_params(**study_lgbm.best_trial.user_attrs['pipeline_params'])
pipeline_lgbm_best.fit(X_train, y_train)

y_pred_proba_lgbm = pipeline_lgbm_best.predict_proba(X_test)

df_test_lgbm = df.loc[X_test.index].copy()
df_test_lgbm['y_pred_proba'] = y_pred_proba_lgbm[:, 1]

ndcg_lgbm_als, precision_lgbm_als, recall_lgbm_als, f1_lgbm_als = calculate_metrics(df_test_lgbm)
metrics_lgbm_als = {
    'NDCG': ndcg_lgbm_als,
    'Precision': precision_lgbm_als,
    'Recall': recall_lgbm_als,
    'F1': f1_lgbm_als
}

Средний NDCG: 0.7817
Средний Precision: 0.6875
Средний Recall: 0.7581
Средний F1-Score: 0.7078


In [106]:
RUN_NAME_LGBM = "best_optuna_lgbm_als"
REGISTRY_MODEL_NAME_LGBM = "best_optuna_lgbm_als"

signature = mlflow.models.infer_signature(X_test, y_test)
input_example = X_test[:10]
code_paths = ["ML_Experiments.ipynb"]

with mlflow.start_run(run_name=RUN_NAME_LGBM, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    
    lgbm_info = mlflow.sklearn.log_model(sk_model=pipeline_lgbm_best, 
                                        artifact_path='best_optuna_lgbm_als',
                                        registered_model_name=REGISTRY_MODEL_NAME_LGBM,
                                        input_example=input_example,
                                        code_paths=code_paths,
                                        await_registration_for=60
                                       )
    mlflow.log_metrics(metrics_lgbm_als)
    mlflow.log_params(best_params_lgbm)

Registered model 'best_optuna_lgbm_als' already exists. Creating a new version of this model...
2026/05/02 19:01:09 INFO mlflow.store.model_registry.abstract_store: Waiting up to 60 seconds for model version to finish creation. Model name: best_optuna_lgbm_als, version 4


🏃 View run best_optuna_lgbm_als at: http://127.0.0.1:5000/#/experiments/1/runs/292b984f101d4b48933f867a07bb8c31
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


Created version '4' of model 'best_optuna_lgbm_als'.


## CatBoostClassifier

In [107]:
def objective_catboost(trial: optuna.Trial) -> float:
    params = {
        'model__iterations': trial.suggest_int('iterations', 100, 1000, step=50),
        'model__depth': trial.suggest_int('depth', 3, 10),
        'model__learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'model__l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10, log=True)
    }
    
    pipeline_catboost = Pipeline([
        ('preprocessing', ColumnTransformer([
            ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
        ], remainder='passthrough')),
        ('model', CatBoostClassifier(
            random_state=RANDOM_STATE, 
            verbose=0, 
            auto_class_weights='Balanced'
        ))
    ])
    
    pipeline_catboost.set_params(**params)
    
    kfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    
    for train_idx, val_idx in kfold.split(X_train, y_train):
        X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        pipeline_catboost.fit(X_fold_train, y_fold_train)
        y_pred_proba = pipeline_catboost.predict_proba(X_fold_val)
        
        df_val = df.loc[X_fold_val.index].copy()
        df_val['y_pred_proba'] = y_pred_proba[:, 1]
        
        ndcg, _, _, _ = calculate_metrics(df_val)
        
        trial.set_user_attr('pipeline_params', params)
    
    return ndcg

In [108]:
RUN_NAME_OPTUNE_CATBOOST = 'CatBoostClassifier_optuna_als'

with mlflow.start_run(run_name=RUN_NAME_OPTUNE_CATBOOST, experiment_id=experiment_id) as run:
    run_id_catboost = run.info.run_id

🏃 View run CatBoostClassifier_optuna_als at: http://127.0.0.1:5000/#/experiments/1/runs/af975241b00d4ad9874914e7c42e402e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [109]:
STUDY_NAME_CATBOOST = "CatBoostClassifier_optuna_als"

mlflc_catboost = MLflowCallback(
    tracking_uri=f'http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}',
    metric_name="NDCG",
    create_experiment=False,
    mlflow_kwargs={'experiment_id': experiment_id, 'tags': {MLFLOW_PARENT_RUN_ID: run_id_catboost}}
)

In [110]:
study_catboost = optuna.create_study(direction='maximize', 
                                     sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
                                     study_name=STUDY_NAME_CATBOOST,
                                     storage=STUDY_DB_NAME,
                                     load_if_exists=True)

study_catboost.optimize(objective_catboost, 
                        n_trials=10, 
                        callbacks=[mlflc_catboost]
                       )

best_params_catboost = study_catboost.best_params
print(f"Number of finished trials: {len(study_catboost.trials)}")
print(f"Best params CatBoost: {best_params_catboost}")

[I 2026-05-02 19:01:09,909] Using an existing study with name 'CatBoostClassifier_optuna_als' instead of creating a new one.


Средний NDCG: 0.8678
Средний Precision: 0.7864
Средний Recall: 0.8426
Средний F1-Score: 0.8028
Средний NDCG: 0.8570
Средний Precision: 0.7804
Средний Recall: 0.8335
Средний F1-Score: 0.7950


[I 2026-05-02 19:01:36,604] Trial 30 finished with value: 0.863096943265915 and parameters: {'iterations': 450, 'depth': 10, 'learning_rate': 0.12562455575707368, 'l2_leaf_reg': 4.036828375507741}. Best is trial 26 with value: 0.8640644662917454.


Средний NDCG: 0.8631
Средний Precision: 0.7880
Средний Recall: 0.8359
Средний F1-Score: 0.8007
🏃 View run 30 at: http://127.0.0.1:5000/#/experiments/1/runs/3d12aa19444c43a6a344e2cc202afda3
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8678
Средний Precision: 0.7857
Средний Recall: 0.8452
Средний F1-Score: 0.8034
Средний NDCG: 0.8575
Средний Precision: 0.7738
Средний Recall: 0.8359
Средний F1-Score: 0.7922


[I 2026-05-02 19:02:00,920] Trial 31 finished with value: 0.8630894080425824 and parameters: {'iterations': 350, 'depth': 10, 'learning_rate': 0.0997594753585949, 'l2_leaf_reg': 6.846757952799266}. Best is trial 26 with value: 0.8640644662917454.


...

Средний NDCG: 0.8632
Средний Precision: 0.7889
Средний Recall: 0.8357
Средний F1-Score: 0.8007
🏃 View run 37 at: http://127.0.0.1:5000/#/experiments/1/runs/acbfa4df981f470aa9a02de8b628cae0
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8679
Средний Precision: 0.7672
Средний Recall: 0.8501
Средний F1-Score: 0.7940
Средний NDCG: 0.8575
Средний Precision: 0.7529
Средний Recall: 0.8398
Средний F1-Score: 0.7808


[I 2026-05-02 19:04:24,206] Trial 38 finished with value: 0.8640661734646502 and parameters: {'iterations': 500, 'depth': 8, 'learning_rate': 0.04869314121216148, 'l2_leaf_reg': 1.2941246721556594}. Best is trial 38 with value: 0.8640661734646502.


Средний NDCG: 0.8641
Средний Precision: 0.7654
Средний Recall: 0.8449
Средний F1-Score: 0.7907
🏃 View run 38 at: http://127.0.0.1:5000/#/experiments/1/runs/c8a2d4557ef44b20856def193cc8d38f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8680
Средний Precision: 0.7634
Средний Recall: 0.8513
Средний F1-Score: 0.7920
Средний NDCG: 0.8576
Средний Precision: 0.7509
Средний Recall: 0.8408
Средний F1-Score: 0.7798


[I 2026-05-02 19:04:45,082] Trial 39 finished with value: 0.8638548804260722 and parameters: {'iterations': 450, 'depth': 8, 'learning_rate': 0.04610633519632583, 'l2_leaf_reg': 1.7617709130227701}. Best is trial 38 with value: 0.8640661734646502.


Средний NDCG: 0.8639
Средний Precision: 0.7647
Средний Recall: 0.8463
Средний F1-Score: 0.7911
🏃 View run 39 at: http://127.0.0.1:5000/#/experiments/1/runs/bb383a8555fa43fe9abd632028130618
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Number of finished trials: 40
Best params CatBoost: {'iterations': 500, 'depth': 8, 'learning_rate': 0.04869314121216148, 'l2_leaf_reg': 1.2941246721556594}


In [111]:
pipeline_catboost_best = Pipeline([
    ('preprocessing', ColumnTransformer([
        ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
    ], remainder='passthrough')),
    ('model', CatBoostClassifier(
        random_state=RANDOM_STATE, 
        verbose=0, 
        auto_class_weights='Balanced'
    ))
])

pipeline_catboost_best.set_params(**study_catboost.best_trial.user_attrs['pipeline_params'])
pipeline_catboost_best.fit(X_train, y_train)

y_pred_proba_catboost = pipeline_catboost_best.predict_proba(X_test)

df_test_catboost = df.loc[X_test.index].copy()
df_test_catboost['y_pred_proba'] = y_pred_proba_catboost[:, 1]

ndcg_catboost_als, precision_catboost_als, recall_catboost_als, f1_catboost_als = calculate_metrics(df_test_catboost)
metrics_catboost_als = {
    'NDCG': ndcg_catboost_als,
    'Precision': precision_catboost_als,
    'Recall': recall_catboost_als,
    'F1': f1_catboost_als
}

Средний NDCG: 0.7813
Средний Precision: 0.6857
Средний Recall: 0.7587
Средний F1-Score: 0.7066


In [112]:
RUN_NAME_CATBOOST = "best_optuna_catboost_als"
REGISTRY_MODEL_NAME_CATBOOST = "best_optuna_catboost_als"

signature = mlflow.models.infer_signature(X_test, y_test)
input_example = X_test[:10]
code_paths = ["ML_Experiments.ipynb"]

with mlflow.start_run(run_name=RUN_NAME_CATBOOST, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    
    catboost_info = mlflow.sklearn.log_model(sk_model=pipeline_catboost_best, 
                                             artifact_path='best_optuna_catboost_als',
                                             registered_model_name=REGISTRY_MODEL_NAME_CATBOOST,
                                             input_example=input_example,
                                             code_paths=code_paths,
                                             await_registration_for=60
                                            )
    mlflow.log_metrics(metrics_catboost_als)
    mlflow.log_params(best_params_catboost)

Registered model 'best_optuna_catboost_als' already exists. Creating a new version of this model...
2026/05/02 19:04:55 INFO mlflow.store.model_registry.abstract_store: Waiting up to 60 seconds for model version to finish creation. Model name: best_optuna_catboost_als, version 4


🏃 View run best_optuna_catboost_als at: http://127.0.0.1:5000/#/experiments/1/runs/8b79fd009f4d40509b2357db017e88c2
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


Created version '4' of model 'best_optuna_catboost_als'.


## Model comparison

In [113]:
models_comparison = pd.DataFrame({
    'Model': ['LogisticRegression', 'RandomForest', 'XGBoost', 'LightGBM', 'CatBoost', 'XGBoost + ALS', 'LightGBM + ALS', 'CatBoost + ALS'],
    'NDCG': [ndcg_lr, ndcg_rf, ndcg_xgb, ndcg_lgbm, ndcg_catboost, ndcg_xgb_als, ndcg_lgbm_als, ndcg_catboost_als],
    'Precision': [precision_lr, precision_rf, precision_xgb, precision_lgbm, precision_catboost, precision_xgb_als, precision_lgbm_als, precision_catboost_als],
    'Recall': [recall_lr, recall_rf, recall_xgb, recall_lgbm, recall_catboost, recall_xgb_als, recall_lgbm_als, recall_catboost_als],
    'F1': [f1_lr, f1_rf, f1_xgb, f1_lgbm, f1_catboost, f1_xgb_als, f1_lgbm_als, f1_catboost_als]
})

models_comparison

,Model,NDCG,Precision,Recall,F1
0,LogisticRegression,0.751483,0.638633,0.598481,0.602124
1,RandomForest,0.776260,0.665618,0.744510,0.686675
2,XGBoost,0.779192,0.696847,0.741136,0.705449
3,LightGBM,0.779915,0.674173,0.752533,0.695656
4,CatBoost,0.779173,0.678703,0.749736,0.697520
5,XGBoost + ALS,0.781168,0.708642,0.742456,0.713721
6,LightGBM + ALS,0.781721,0.687478,0.758067,0.707797
7,CatBoost + ALS,0.781273,0.685669,0.758717,0.706570


In [114]:
best_model_idx = models_comparison['NDCG'].idxmax()
print(f"\nЛучшая модель: {models_comparison.iloc[best_model_idx]['Model']} с NDCG = {models_comparison.iloc[best_model_idx]['NDCG']:.4f}")


Лучшая модель: LightGBM + ALS с NDCG = 0.7817


<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

Сохраним обученный итоговый пайплайн.

</div>

In [116]:
MODEL_NAME = 'pipeline_lgb_best.pkl'
with open(MODEL_NAME, 'wb') as file:
    pickle.dump(pipeline_lgbm_best, file)